<div align="center">

# Hafta 11: Final projesi ve sertifika

**Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı**<br>
Türk Kardiyoloji Derneği · Prof. Dr. Utku Köse

[Hafta sayfası](README.md) · [Ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-11/ders.html) · [Etkileşimli lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-11/lab.html) · [PDF ders notu](Hafta11_Ders_Notlari.pdf) · [Colab'da aç](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-11/NB11_final_projesi_atolyesi.ipynb)

</div>

**Bu defter iki kısımdan oluşur.** Birincisinde kodu bir yapay zekâ (YZ) asistanı yazar; asistana ne yapacağını yazılı bir talimat, yani bir prompt (istem) anlatır. İkincisinde kodu Python ile kendiniz yazarsınız. İki kısım da aynı verilerle aynı sonuçlara varır; ders sayfasındaki her uygulama durağında bunlardan birini seçin.

| | Kimler için? | Ne yapılır? |
|---|---|---|
| **[Kısım 1: Prompt yolu](#part-1)** | Kod yazmayan katılımcılar | Kodu, hazır promptlarla bir YZ asistanı yazar; siz çalıştırır ve sonucu denetlersiniz. |
| **[Kısım 2: Python yolu](#part-2)** | Python öğrenmek isteyen katılımcılar | Kodu alıştırmalarla adım adım kendiniz yazar ve çalıştırırsınız. |

**Nerede çalışır?** Aynı defter, hiçbir kurulum gerektirmeyen Google Colab'da da kendi bilgisayarınızdaki Jupyter'de de (Anaconda; ders klasöründeki SETUP.tr.md dosyasına bakın) çalışır. Hangisini tercih ederseniz onu kullanın. Colab'da önce *Dosya → Drive'a kopya kaydet* (*File → Save a copy in Drive*) komutuyla kendi kopyanızı kaydedin; aksi hâlde sekme kapanınca yaptığınız değişiklikler kaybolur. 2. kısmın her bölümü sıradan bir dizüstü bilgisayarda birkaç saniyede çalışır; 5. bölümdeki bootstrap yöntemi 2.000 yeniden örneklemi yaklaşık on saniyede çeker. Defter kendi yanına sonuçları, bir grafiği ve final projesinin araç takımını içeren final_project klasörünü yazar. Hangi kısmı seçerseniz seçin önce 0. bölümü çalıştırın. Bu bölüm ders dosyalarını ve paketleri hazırlar.

**Veri ve mahremiyet.** Buradaki bütün veri setleri ya sentetiktir ya da açık bir araştırma veri tabanından gelmektedir. Gerçek hasta verisi yalnızca kendi bilgisayarınızda, kurumunuzun kurallarına uygun olarak ve etik kurul onayıyla işlenir; bulut tabanlı bir deftere ya da bir YZ sohbetine asla konmaz.

## 0. Kurulum

İlk hücre ders klasörünü bulur. Defter tek başına açıldıysa, örneğin Google Colab'da, `vclinic` ders paketini ve veri dosyalarını GitHub'dan indirir. İkinci hücre paketleri yükler, mesajların dilini seçer ve haftanın isteğe bağlı paketlerini denetler. Cleveland verisi ders klasörüyle birlikte geldiği için bu hafta hiçbir şey indirilmez.

In [ ]:
#@title Ders dosyaları: Bu hücreyi çalıştırın (açmanıza gerek yok)
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
DATA_FILES = [
    "data/uci-heart-disease/processed.cleveland.data",
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None or ROOT.name == "ai-cardiology-course":
    # İndirilmiş bir kopyada, örneğin Colab'da, paketin dosyaları yenilenir.
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course" if ROOT is None else ROOT
    manifest = urllib.request.urlopen(REPO_RAW + "vclinic/MANIFEST.txt").read().decode("utf-8")
    package = ["vclinic/" + line.strip() for line in manifest.splitlines() if line.strip()]
    for rel in package + DATA_FILES:
        target = ROOT / rel
        if rel in package or not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Ders dosyaları şu klasöre indirildi:", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# Kurulum: paketler, dil ve çizim biçimi
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json
import math
import pathlib
import platform
import sys

import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vclinic.datasets import load_cleveland
import vclinic
from vclinic import teach

if tuple(int(x) for x in vclinic.__version__.split(".")) < (1, 10, 0):
    raise ImportError("vclinic ders paketi bu defterden daha eskidir. Ders klasörünü güncelleyin (git pull komutuyla ya da ZIP dosyasını yeniden indirerek) ve çekirdeği yeniden başlatın.")
teach.set_language("tr")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})
RANDOM_STATE = 2026

print("vclinic sürümü", vclinic.__version__)
print("Ders klasörü:", ROOT)
for name in []:
    try:
        importlib.import_module(name)
        print(f"{name}: kurulu")
    except ImportError:
        print(f"{name}: kurulu değil (isteğe bağlı; defter yerleşik bir yedek yol kullanır)")

In [ ]:
#@title Alıştırmaların beklenen cevapları (bu hücreyi çalıştırın; açmayı alıştırmaları denedikten sonraya bırakın)
teach.register_answers({
    "Alıştırma 11.1": (6, 0),
    "Alıştırma 11.2": (5, 0),
    "Alıştırma 11.3": (0.625, 0.001),
    "Alıştırma A": (0.459, 0.001),
    "Alıştırma B": (35, 0),
    "Alıştırma C": (121, 0),
    "Alıştırma D": (0.928, 0.005),
    "Alıştırma E": (0.129, 0.02),
    "Alıştırma F": (1.523, 0.01),
    "Alıştırma G": (0.26, 0),
    "Alıştırma H": (0.155, 0.005),
    "Alıştırma I": (5, 0),
    "Alıştırma J": (0.753, 0.005),
    "Adım 1.1": (6, 0),
    "Adım 1.2": (241, 2),
    "Adım 1.3": (0.9, 0.01),
    "Adım 2.1": (0.872, 0.01),
    "Adım 2.2": (-0.02, 0.015),
    "Adım 2.3": (1.241, 0.01),
    "Adım 3.1": (0.18, 0),
    "Adım 3.2": (27, 0),
    "Adım 3.3": (0.646, 0.005),
})
print("Beklenen cevaplar hazır.")

In [ ]:
#@title Alıştırma promptlarının tabloları (bu hücreyi çalıştırın; Python yolunda okunması gerekmez)
def alistirma_tablosu(anahtar):
    """Bir alıştırma promptunun "veri" tablosu; haftanın verilerinden kurulur."""
    g = globals()
    k = anahtar
    need = lambda *names: [g[n] for n in names] if all(n in g for n in names) else (_ for _ in ()).throw(
        RuntimeError("Önce bu alıştırmadan önceki bölümleri çalıştırın."))
    cp = {1.0: "tipik anjina", 2.0: "atipik anjina", 3.0: "anjina dışı ağrı", 4.0: "ağrı yok"}
    if k == "A":
        heart, = need("heart")
        return pd.DataFrame({"hasta": heart.index, "hastalık": heart["disease"]}).reset_index(drop=True)
    if k in ("B", "D"):
        heart, X_train, X_test = need("heart", "X_train", "X_test")
        idx = list(X_train.index) + list(X_test.index)
        h = heart.loc[idx]
        t = pd.DataFrame({"hasta": h.index, "küme": ["eğitim"] * len(X_train) + ["test"] * len(X_test)})
        if k == "D":
            t = t.assign(**{"yaş": h["age"].astype(int).to_numpy(), "cinsiyet": h["sex"].map({1.0: "erkek", 0.0: "kadın"}).to_numpy(),
                            "göğüs ağrısı tipi": h["cp"].map(cp).to_numpy(), "en yüksek kalp hızı": h["thalach"].astype(int).to_numpy(),
                            "egzersiz anjinası": h["exang"].astype(int).to_numpy(), "ST çökmesi": h["oldpeak"].to_numpy()})
        return t.assign(**{"hastalık": h["disease"].to_numpy()})
    if k in ("E", "F"):
        X_test, yt, p_ref, p_model = need("X_test", "yt", "p_ref", "p_model")
        t = pd.DataFrame({"hasta": X_test.index, "hastalık": yt, "olasılık, referans": np.round(p_ref, 6)})
        return t.assign(**{"olasılık, model": np.round(p_model, 6)}) if k == "E" else t
    if k == "G":
        X_train, y_train, p_cv = need("X_train", "y_train", "p_cv")
        return pd.DataFrame({"hasta": X_train.index, "hastalık": y_train.to_numpy(),
                             "olasılık, çapraz doğrulama": np.round(p_cv, 6)})
    if k == "H":
        low, scaler, logistic, ALL = need("low", "scaler", "logistic", "ALL")
        names = {"age": "yaş", "sex": "erkek cinsiyet", "trestbps": "istirahat kan basıncı", "chol": "kolesterol",
                 "thalach": "en yüksek kalp hızı", "exang": "egzersiz anjinası", "oldpeak": "ST çökmesi",
                 "cp_2.0": "atipik anjina", "cp_3.0": "anjina dışı ağrı", "cp_4.0": "ağrı yok"}
        return pd.DataFrame({"sütun": [names[c] for c in ALL] + ["sabit terim"],
                             "hastanın değeri": [float(low[c].iloc[0]) for c in ALL] + [np.nan],
                             "ortalama, eğitim": list(np.round(scaler.mean_, 4)) + [np.nan],
                             "standart sapma, eğitim": list(np.round(scaler.scale_, 4)) + [np.nan],
                             "katsayı": list(np.round(logistic.coef_[0], 4)) + [round(float(logistic.intercept_[0]), 4)]})
    if k in ("I", "J"):
        X_test, yt, p_model = need("X_test", "yt", "p_model")
        t = pd.DataFrame({"hasta": X_test.index})
        if k == "I":
            t["yaş"] = X_test["age"].astype(int).to_numpy()
        return t.assign(**{"hastalık": yt, "olasılık, model": np.round(p_model, 6)})
    raise KeyError(k)
print("Alıştırma promptlarının tabloları hazır.")

<a id="part-1"></a>

# Kısım 1: Prompt yolu

**Kimler için?** Bu kısım kod yazmayan katılımcılar içindir. Burada hiç kod yazmazsınız. Kodu, hazır bir prompttan, yani yazılı bir talimattan yararlanarak bir yapay zekâ (YZ) asistanı yazar; siz kodu çalıştırır ve sonucu denetlersiniz.

**Neye ihtiyacınız var?** Python kodu yazabilen bir YZ asistanı yeterlidir. Google Colab'ın içinde Gemini bulunur; pencerenin üstündeki Gemini düğmesiyle açılır. Başka bir tarayıcı sekmesinde açılan ChatGPT, Claude, Copilot ya da Gemini de aynı işi görür.

**Bir adım nasıl ilerler?**

1. Adımın amacını okuyun.
2. Promptun altındaki veri hücresini çalıştırın. Hücre, promptun tarif ettiği "veri" tablosunu hazırlar ve tablonun ilk ve son satırlarını gösterir.
3. Promptu kopyalayın: Fareyle seçin ve Ctrl+C'ye (Mac'te Cmd+C) basın.
4. Promptu YZ asistanına yapıştırın ve gönderin.
5. Asistanın verdiği kodu kopyalayın, veri hücresinin altındaki hücreye `#` ile başlayan satırın yerine yapıştırın ve hücreyi ▶ düğmesiyle ya da Shift+Enter ile çalıştırın.
6. Hemen altındaki kontrol hücresini çalıştırın. Hücre *doğru ✓* ya da *henüz değil ✗* yazar.
7. *Henüz değil* yazarsa aşağıdaki onarım promptunu kodla birlikte asistana gönderin. Sonuç yine yanlışsa *Hazır kod* hücresini çalıştırın. Bu hücre her zaman doğru sonucu verir; böylece bir sonraki adıma geçebilirsiniz.

**"veri" ve "cevap".** 1. haftada olduğu gibi her adımın veri hücresi, promptun tarif ettiği "veri" tablosunu hazırlar. Asistan sonucu "cevap" adıyla saklar; kontrol hücresi de sonucu bu addan okur.

**Her durağın başında** o durağın hazırlık hücresini bir kez çalıştırın. Hücre durağın verilerini yükler; böylece bir durağa başka bir gün de başlanabilir.

**Onarım promptu** (çalışan ama kontrolden geçmeyen kod için):

> Aşağıdaki kod şu görev için yazıldı: [promptun Görev satırını yapıştırın]. Kod çalışıyor, ancak dersin kontrolü "henüz değil" yazıyor. Hatayı bul, iki cümleyle açıkla ve düzeltilmiş kodu ver. Sonuç yine "cevap" adıyla saklanmalı.  
>
> [kodu buraya yapıştırın]  

**Hata mesajı için prompt** (hata vererek duran kod için):

> Aşağıdaki kod bir Jupyter defterinde çalıştırıldığında altta verilen hata mesajını veriyor. Hatanın nedenini sade bir dille açıkla ve düzeltilmiş kodu ver.  
>
> [kodu buraya yapıştırın]  
>
> [hata mesajını buraya yapıştırın]  

Haftanın bütün promptları [prompt kütüphanesinde](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-11/PROMPTLAR.md) de yer alır; GitHub her promptun yanında bir kopyalama düğmesi gösterir.

**Hiç değişmeyen kural.** Promptlarda yalnızca dersin sentetik ya da açık verileri bulunur. Bir YZ asistanına gerçek hasta verisi asla yapıştırılmaz.

## Durak 1: Soru, veriler ve bir referans

*Ders: [1. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-11/ders.html#stop-1)* · *Python yolunda aynı sonuç: Kısım 2, 1-4. bölümler*

Bu durak, haftanın çözümlü projesi üzerinde final projeleri atölyesinin ilk yarısıdır. Cleveland hastalarının verileri denetlenir, on parametreli bir modelin ihtiyaç duyduğu büyüklük eğitim hastalarıyla karşılaştırılır ve yaş, cinsiyet ve göğüs ağrısı tipinden oluşan referans test hastalarında bir kez sınanır. Duraktan sonra aynı üç adım kendi final projenizin verileriyle tekrarlanır.

**Durağın sonunda:** Projenin dışarıda bıraktığı sütunlardaki eksik değerleri (6), Riley ve arkadaşlarının (i) ölçütünün istediği büyüklüğü (241 hasta) ve referansın test AUC değerini (0,900) bulacaksınız.

In [ ]:
#@title 1. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
_CP = {1.0: "tipik anjina", 2.0: "atipik anjina", 3.0: "anjina dışı ağrı", 4.0: "ağrı yok"}


def _course():
    """Haftanın çözümlü projesi: Cleveland verisi, 2. haftanın ayırması, referans, tam model ve eğitim hastalarının çapraz doğrulama olasılıkları (bir kez hesaplanır, birkaç saniye sürer)."""
    if "_course_cache" in globals():
        return _course_cache
    heart = load_cleveland()
    FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
    missing = heart.isna().sum()
    audit = pd.DataFrame({"eksik değer": missing, "projede kullanılıyor": [c in FEATURES for c in missing.index]})

    cad = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"], drop_first=True, dtype=int)
    X, y = cad.drop(columns="disease"), cad["disease"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)


    def cox_snell_r2(y, p):
        """Tahmin edilen p olasılıklarının y sonuçları için Cox-Snell R kare değerini hesaplar."""
        y, p = np.asarray(y, dtype=float), np.clip(np.asarray(p, dtype=float), 1e-12, 1 - 1e-12)
        phi = y.mean()
        ll_null = len(y) * (phi * np.log(phi) + (1 - phi) * np.log(1 - phi))
        ll_model = np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
        return 1 - np.exp(2 * (ll_null - ll_model) / len(y))


    def riley_sizes(P, prevalence, r2_cs, shrinkage=0.9, delta=0.05, margin=0.05):
        """Riley ve arkadaşlarının P parametreli bir ikili sonuç modeli için önerdiği üç büyüklüğü hesaplar. Ölçütler (i) en az 0,9 büzülme, (ii) görünen ve düzeltilmiş Nagelkerke R kare değerleri arasında en fazla 0,05 fark ve (iii) genel riskin 0,05 içinde kestirilmesidir."""
        r2_max = 1 - np.exp(2 * (prevalence * np.log(prevalence) + (1 - prevalence) * np.log(1 - prevalence)))
        n1 = P / ((shrinkage - 1) * np.log(1 - r2_cs / shrinkage))
        s2 = r2_cs / (r2_cs + delta * r2_max)
        n2 = P / ((s2 - 1) * np.log(1 - r2_cs / s2))
        n3 = (1.96 / margin) ** 2 * prevalence * (1 - prevalence)
        return {"(i)": math.ceil(n1), "(ii)": math.ceil(n2), "(iii)": math.ceil(n3), "R2_max": round(float(r2_max), 3)}

    REFERENCE = ["age", "sex", "cp_2.0", "cp_3.0", "cp_4.0"]
    anticipated = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[REFERENCE], y_train)
    r2_cs = cox_snell_r2(y_train, anticipated.predict_proba(X_train[REFERENCE])[:, 1])
    P = X_train.shape[1]
    sizes = riley_sizes(P, float(y_train.mean()), r2_cs)

    ALL = list(X.columns)


    def fit(columns):
        """Eğitim hastalarında bir lojistik regresyon kurar; ölçekleme işlem hattının içinde yer alır ve yalnızca eğitim hastalarından öğrenilir."""
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[columns], y_train)


    reference, model = fit(REFERENCE), fit(ALL)
    p_ref = reference.predict_proba(X_test[REFERENCE])[:, 1]
    p_model = model.predict_proba(X_test[ALL])[:, 1]
    yt = y_test.to_numpy()

    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    p_cv = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X_train[ALL], y_train, cv=folds, method="predict_proba")[:, 1]
    GRID = np.round(np.arange(0.01, 0.51, 0.01), 2)
    threshold = max(t for t in GRID if (p_cv[y_train.to_numpy() == 1] >= t).mean() >= 0.95)
    ruled_out = p_model < threshold
    counts = {"TP": int((~ruled_out & (yt == 1)).sum()), "FN": int((ruled_out & (yt == 1)).sum()),
              "TN": int((ruled_out & (yt == 0)).sum()), "FP": int((~ruled_out & (yt == 0)).sum())}
    p_ref_train = reference.predict_proba(X_train[REFERENCE])[:, 1]
    globals()["_course_cache"] = dict(heart=heart, X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test, p_ref=p_ref,
                                      p_model=p_model, p_cv=p_cv, p_ref_train=p_ref_train, threshold=threshold)
    return _course_cache


def _patients(which):
    """Eğitim ya da test hastaları veya ayırmanın sırasıyla ikisi birden; kümeleri, sekiz klinik değişken ve hastalıkla birlikte."""
    c = _course()
    parts = {"eğitim": c["X_train"].index, "test": c["X_test"].index}
    rows = [(s, i) for s, idx in parts.items() for i in idx if which == "both" or s == which]
    h = c["heart"].loc[[i for _, i in rows]]
    return pd.DataFrame({"hasta": h.index, "küme": [s for s, _ in rows],
                         "yaş": h["age"].astype(int), "cinsiyet": h["sex"].map({1.0: "erkek", 0.0: "kadın"}),
                         "göğüs ağrısı tipi": h["cp"].map(_CP),
                         "istirahat kan basıncı": h["trestbps"].astype(int), "kolesterol": h["chol"].astype(int),
                         "en yüksek kalp hızı": h["thalach"].astype(int), "egzersiz anjinası": h["exang"].astype(int),
                         "ST çökmesi": h["oldpeak"], "hastalık": h["disease"]}).reset_index(drop=True)


_c = _course()
print(len(_c["heart"]), "hasta Cleveland verisinde:", len(_c["X_train"]), "eğitim ve", len(_c["X_test"]), "test için")
print("1. durak hazır.")

### Adım 1.1: Verilerin denetimi

**Amaç.** Bu adımda verilerin eksik değerleri sütun sütun sayılır ve projenin hangi sütunları kullandığı görülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Bir kardiyolog, bazı hastaların anjiyografiye gönderilmesine gerek kalmasın diye egzersiz testinden sonra koroner arter hastalığını dışlamayı amaçlayan bir final projesi planlıyor. Herhangi bir modelden önce veriler denetlenir: Hangi sütunlarda eksik değer var ve bunlar kararın kendi anında sahip olacağı sütunlar mı?  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda UCI Heart Disease veri setinin Cleveland verisinden, yani bir egzersiz testinden sonra koroner anjiyografiye gönderilmiş hastalardan 303 hasta, bütün sütunlarıyla birlikte bulunur. Her satır bir hastadır: "hasta" hastanın verideki numarasını (0-302) verir. Projenin sekiz değişkeni şunlardır: "yaş" (yıl), "cinsiyet" ("erkek" ya da "kadın"), "göğüs ağrısı tipi" ("tipik anjina", "atipik anjina", "anjina dışı ağrı" ya da "ağrı yok"), "istirahat kan basıncı" (mmHg), "kolesterol" (mg/dL), "en yüksek kalp hızı" (egzersiz testinde ulaşılan en yüksek kalp hızı, dakikada), "egzersiz anjinası" (egzersiz testi anjinaya yol açtıysa 1) ve "ST çökmesi" (egzersizde ST segmentinin istirahate göre çökmesi, mm). Veri setinden beş sütun daha gelir: "açlık kan şekeri 120 üstü" (açlık kan şekeri 120 mg/dL'nin üstündeyse 1), "istirahat EKG'si" (0 normal, 1 ST segmenti ya da T dalgası bozukluğu, 2 sol ventrikül hipertrofisi), "ST eğimi" (egzersizin doruğunda ST segmentinin eğimi: 1 yukarı, 2 düz, 3 aşağı), "floroskopide damar" (floroskopiyle boyanan ana damar sayısı, 0-3) ve "talyum sintigrafisi" (3 normal, 6 sabit defekt, 7 geri dönüşlü defekt). Boş bir hücre eksik bir değerdir. "hastalık", anjiyogramda en az bir ana koroner arterde çapın yüzde 50'sinden fazla daralma görüldüyse 1, aksi hâlde 0 olur.  
> Görev: "hasta" dışındaki her sütunun eksik değerlerini say. Proje dokuz sütun kullanır; bunlar sekiz değişken ve "hastalık" sütunudur. Her sütun için projenin onu kullanıp kullanmadığını işaretle ve projenin kullanmadığı sütunlardaki eksik değerleri topla. Ayrıca hasta sayısını, hastalığı olanların sayısını ve oranlarını ver.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her sütunun eksik değerlerini ve projenin o sütunu kullanıp kullanmadığını bir tabloda göster, hasta sayılarını yazdır; projenin kullanmadığı sütunlardaki eksik değer sayısını "cevap" adıyla sakla.  
> Test: Projenin kullandığı dokuz sütunda hiç eksik değer olmamalı ve tablo 303 hasta içermeli.  

In [ ]:
#@title Adım 1.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_h = _course()["heart"]
veri = pd.DataFrame({"hasta": _h.index, "yaş": _h["age"], "cinsiyet": _h["sex"].map({1.0: "erkek", 0.0: "kadın"}),
                     "göğüs ağrısı tipi": _h["cp"].map(_CP), "istirahat kan basıncı": _h["trestbps"],
                     "kolesterol": _h["chol"], "açlık kan şekeri 120 üstü": _h["fbs"],
                     "istirahat EKG'si": _h["restecg"], "en yüksek kalp hızı": _h["thalach"],
                     "egzersiz anjinası": _h["exang"], "ST çökmesi": _h["oldpeak"], "ST eğimi": _h["slope"],
                     "floroskopide damar": _h["ca"], "talyum sintigrafisi": _h["thal"],
                     "hastalık": _h["disease"]}).reset_index(drop=True)
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.1 kontrolü
teach.check_step("Adım 1.1", "cevap", globals())

In [ ]:
#@title Adım 1.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_h = _course()["heart"]
veri = pd.DataFrame({"hasta": _h.index, "yaş": _h["age"], "cinsiyet": _h["sex"].map({1.0: "erkek", 0.0: "kadın"}),
                     "göğüs ağrısı tipi": _h["cp"].map(_CP), "istirahat kan basıncı": _h["trestbps"],
                     "kolesterol": _h["chol"], "açlık kan şekeri 120 üstü": _h["fbs"],
                     "istirahat EKG'si": _h["restecg"], "en yüksek kalp hızı": _h["thalach"],
                     "egzersiz anjinası": _h["exang"], "ST çökmesi": _h["oldpeak"], "ST eğimi": _h["slope"],
                     "floroskopide damar": _h["ca"], "talyum sintigrafisi": _h["thal"],
                     "hastalık": _h["disease"]}).reset_index(drop=True)
d = veri
kullanilan = ["yaş", "cinsiyet", "göğüs ağrısı tipi", "istirahat kan basıncı", "kolesterol",
          "en yüksek kalp hızı", "egzersiz anjinası", "ST çökmesi", "hastalık"]
eksik = d.drop(columns="hasta").isna().sum()
print(pd.DataFrame({"eksik değer": eksik, "projede kullanılıyor": [c in kullanilan for c in eksik.index]}))
print("hasta", len(d), "| hastalıklı", int(d["hastalık"].sum()), "| oran", round(float(d["hastalık"].mean()), 3))
cevap = int(eksik[[c for c in eksik.index if c not in kullanilan]].sum())
teach.check_step("Adım 1.1", "cevap", globals())

**Sonuç ne gösteriyor?** 6 eksik değerin hepsi floroskopide damar ve talyum sintigrafisi sütunlarındadır. İkisi de egzersiz testinden sonra yapılan tetkiklerin sonucudur; bu yüzden proje onları eksik değerler yüzünden değil, zamanlama yüzünden dışarıda bırakır. Bu sütunları kullanan bir model, sevk kararının henüz sahip olmadığı bir bilgiyi kullanmış olurdu.

### Adım 1.2: Modelin ihtiyaç duyduğu veri

**Amaç.** Bu adımda Riley ve arkadaşlarının ölçütlerine göre on parametreli bir modelin ihtiyaç duyduğu üç en küçük büyüklük hesaplanır ve eğitim hastalarının sayısıyla karşılaştırılır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Projenin tam modelinin on parametresi vardır: Yedi değişken ve göğüs ağrısı tipi için üç sütun. Kardiyolog 227 eğitim hastasının yeterli olup olmadığını öğrenmek istiyor ve öngörülen değer olarak yaş, cinsiyet ve göğüs ağrısı tipinden oluşan referansın R kare değerini alıyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda UCI Heart Disease veri setinin Cleveland verisinden, yani bir egzersiz testinden sonra koroner anjiyografiye gönderilmiş hastalardan 227 eğitim hastası bulunur. Her satır bir hastadır: "hasta" hastanın verideki numarasını, "olasılık, referans" ise dersin bir referans modelinin, yani yaş, cinsiyet ve göğüs ağrısı tipiyle kurulan bir lojistik regresyonun, aynı hastalarda kurulduktan sonra hastaya verdiği koroner arter hastalığı olasılığını verir. "hastalık", anjiyogramda en az bir ana koroner arterde çapın yüzde 50'sinden fazla daralma görüldüyse 1, aksi hâlde 0 olur.  
> Görev: Olasılıkların Cox-Snell R kare değerini hesapla: Olasılıkların log-olabilirliğini, yani her hastada hastanın sahip olduğu sonuca verilen olasılığın doğal logaritmasının toplamını ve her hastaya genel hastalık oranını veren bir modelin log-olabilirliğini al; R kare, boş modelin değerinden modelin değeri çıkarılarak bulunan farkın iki katının hasta sayısına bölümünün üstelinin birden çıkarılmasıyla elde edilir. Ardından Riley ve arkadaşlarının 10 parametreli bir model için üç en küçük büyüklüğünü hesapla. Hastalık oranının izin verdiği en büyük R kare, genel oranın log-olabilirliğinin iki katının hasta sayısına bölümünün üstelinin birden çıkarılmasıyla bulunur. 0,9 büzülme için (i) büyüklüğü, 10'un, (0,9 eksi 1) ile bir eksi R karenin 0,9'a bölümünün doğal logaritmasının çarpımına bölümüdür. (ii) büyüklüğü aynı formülü kullanır; yalnızca büzülmenin yerine R karenin, R kare ile en büyük R karenin 0,05 katının toplamına bölümü konur. (iii) büyüklüğü (1,96 bölü 0,05) karesinin hastalık oranıyla ve bir eksi oranla çarpımıdır. Her büyüklüğü tam bir hastaya yukarı yuvarla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Cox-Snell R kare değerini, en büyük R kareyi, üç büyüklüğü ve eğitim hastası sayısını yazdır; (i) büyüklüğünü "cevap" adıyla sakla.  
> Test: R kare 0 ile en büyük R kare arasında olmalı ve (iii) büyüklüğü olasılıklara bağlı olmamalı.  

In [ ]:
#@title Adım 1.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_c = _course()
veri = pd.DataFrame({"hasta": _c["X_train"].index, "hastalık": _c["y_train"].to_numpy(),
                     "olasılık, referans": np.round(_c["p_ref_train"], 6)})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.2 kontrolü
teach.check_step("Adım 1.2", "cevap", globals())

In [ ]:
#@title Adım 1.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_c = _course()
veri = pd.DataFrame({"hasta": _c["X_train"].index, "hastalık": _c["y_train"].to_numpy(),
                     "olasılık, referans": np.round(_c["p_ref_train"], 6)})
y = veri["hastalık"].to_numpy(dtype=float)
p = veri["olasılık, referans"].to_numpy(dtype=float)
n, phi, P = len(y), float(y.mean()), 10
ll_bos = n * (phi * np.log(phi) + (1 - phi) * np.log(1 - phi))
ll_model = float(np.sum(y * np.log(p) + (1 - y) * np.log(1 - p)))
r2 = 1 - np.exp(2 * (ll_bos - ll_model) / n)
r2_maks = 1 - np.exp(2 * ll_bos / n)
n1 = math.ceil(P / ((0.9 - 1) * np.log(1 - r2 / 0.9)))
s2 = r2 / (r2 + 0.05 * r2_maks)
n2 = math.ceil(P / ((s2 - 1) * np.log(1 - r2 / s2)))
n3 = math.ceil((1.96 / 0.05) ** 2 * phi * (1 - phi))
print("Cox-Snell R kare", round(float(r2), 3), "| en büyük R kare", round(float(r2_maks), 3))
print("büyüklükler (i), (ii), (iii):", n1, n2, n3, "| eğitim hastası", n)
cevap = int(n1)
teach.check_step("Adım 1.2", "cevap", globals())

**Sonuç ne gösteriyor?** (i) ölçütü 241 hasta, (iii) ölçütü daha da fazlasını ister; projede ise 227 hasta vardır. Proje yine de sürdürülür, ancak raporda verilerin ölçütlerin istediğinden küçük olduğu belirtilir; 2. duraktaki kalibrasyon eğimi de bunun getirdiği hafif aşırı öğrenmeyi gösterecektir.

### Adım 1.3: Test hastalarında referans

**Amaç.** Bu adımda yaş, cinsiyet ve göğüs ağrısı tipinden oluşan referans model eğitim hastalarında kurulur ve test hastalarında bir kez sınanır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kardiyolog egzersiz testinin sonuçlarının yardımcı olup olmadığını sormadan önce, projenin aşması gereken çıtaya ihtiyacı vardır. Bu çıta, kardiyoloğun testten önce bildiği yaş, cinsiyet ve göğüs ağrısı tipidir.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda UCI Heart Disease veri setinin Cleveland verisinden, yani bir egzersiz testinden sonra koroner anjiyografiye gönderilmiş hastalardan 303 hasta bulunur. Her satır bir hastadır: "hasta" hastanın verideki numarasını (0-302), "küme" 2. haftanın kümesini verir ("eğitim", 227 hasta; "test", 76 hasta). Klinik değişkenler şunlardır: "yaş" (yıl), "cinsiyet" ("erkek" ya da "kadın"), "göğüs ağrısı tipi" ("tipik anjina", "atipik anjina", "anjina dışı ağrı" ya da "ağrı yok"), "istirahat kan basıncı" (mmHg), "kolesterol" (mg/dL), "en yüksek kalp hızı" (egzersiz testinde ulaşılan en yüksek kalp hızı, dakikada), "egzersiz anjinası" (egzersiz testi anjinaya yol açtıysa 1) ve "ST çökmesi" (egzersizde ST segmentinin istirahate göre çökmesi, mm). "hastalık", anjiyogramda en az bir ana koroner arterde çapın yüzde 50'sinden fazla daralma görüldüyse 1, aksi hâlde 0 olur.  
> Görev: Referansın girdilerini hazırla. Girdiler yaş, erkekler için 1 ve kadınlar için 0 olan bir sütun ve atipik anjina, anjina dışı ağrı ve ağrı yok için 0 ile 1 değerlerinden oluşan üç sütundur; böylece tipik anjina referans kategori olur. Her girdiyi eğitim hastalarının ortalaması ve standart sapmasıyla bir z-skoruna dönüştür ve hastalık için olağan lojistik regresyonu, standart ayarlarıyla ve en fazla 1.000 yinelemeyle, yalnızca eğitim hastalarında kur. Her test hastası için hastalık olasılığını ve test hastalarındaki ROC eğrisi altındaki alanı (AUC) hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Eğitim ve test hastalarının sayısını ve referansın test AUC değerini yazdır; AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Model yalnızca 227 eğitim hastasında kurulmalı ve AUC 0,5 ile 1 arasında olmalı.  

In [ ]:
#@title Adım 1.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _patients("both")
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.3 kontrolü
teach.check_step("Adım 1.3", "cevap", globals())

In [ ]:
#@title Adım 1.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _patients("both")
d = veri
X = pd.DataFrame({"yaş": d["yaş"], "erkek": (d["cinsiyet"] == "erkek").astype(int)})
for c in ["atipik anjina", "anjina dışı ağrı", "ağrı yok"]:
    X[c] = (d["göğüs ağrısı tipi"] == c).astype(int)
eg_ = (d["küme"] == "eğitim").to_numpy()
y = d["hastalık"].to_numpy()
referans = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X[eg_], y[eg_])
p = referans.predict_proba(X[~eg_])[:, 1]
print("eğitim hastası", int(eg_.sum()), "| test hastası", int((~eg_).sum()))
print("referansın test AUC değeri", round(float(roc_auc_score(y[~eg_], p)), 3))
cevap = round(float(roc_auc_score(y[~eg_], p)), 3)
teach.check_step("Adım 1.3", "cevap", globals())

**Sonuç ne gösteriyor?** Referans, her kardiyoloğun testten önce bildiği üç değişkenle testte 0,900 AUC değerine ulaşır. Tam modelin her kazancı 2. durakta bu sayıya karşı ölçülür.

**1. durak tamamlandı.** [Ders sayfasına](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-11/ders.html#stop-1) dönün ve duraktan sonrasını okumaya devam edin.

## Durak 2: Belirsizlik ve kalibrasyon

*Ders: [2. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-11/ders.html#stop-2)* · *Python yolunda aynı sonuç: Kısım 2, 5-6. bölümler*

Bu durak sonuçlardan ne kadar emin olunabileceğini sorar. Bootstrap yöntemi, referansın ve tam modelin AUC değerlerinin ve aynı yeniden örneklemlerdeki farklarının aralıklarını verir; kalibrasyon eğimi ise tam modelin olasılıklarının test hastalarında tutup tutmadığını sorar.

**Durağın sonunda:** Tam modelin test AUC değerinin alt sınırını (0,872), referansla farkının alt sınırını (−0,020) ve test hastalarındaki kalibrasyon eğimini (1,241) bulacaksınız.

In [ ]:
#@title 2. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
_CP = {1.0: "tipik anjina", 2.0: "atipik anjina", 3.0: "anjina dışı ağrı", 4.0: "ağrı yok"}


def _course():
    """Haftanın çözümlü projesi: Cleveland verisi, 2. haftanın ayırması, referans, tam model ve eğitim hastalarının çapraz doğrulama olasılıkları (bir kez hesaplanır, birkaç saniye sürer)."""
    if "_course_cache" in globals():
        return _course_cache
    heart = load_cleveland()
    FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
    missing = heart.isna().sum()
    audit = pd.DataFrame({"eksik değer": missing, "projede kullanılıyor": [c in FEATURES for c in missing.index]})

    cad = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"], drop_first=True, dtype=int)
    X, y = cad.drop(columns="disease"), cad["disease"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)


    def cox_snell_r2(y, p):
        """Tahmin edilen p olasılıklarının y sonuçları için Cox-Snell R kare değerini hesaplar."""
        y, p = np.asarray(y, dtype=float), np.clip(np.asarray(p, dtype=float), 1e-12, 1 - 1e-12)
        phi = y.mean()
        ll_null = len(y) * (phi * np.log(phi) + (1 - phi) * np.log(1 - phi))
        ll_model = np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
        return 1 - np.exp(2 * (ll_null - ll_model) / len(y))


    def riley_sizes(P, prevalence, r2_cs, shrinkage=0.9, delta=0.05, margin=0.05):
        """Riley ve arkadaşlarının P parametreli bir ikili sonuç modeli için önerdiği üç büyüklüğü hesaplar. Ölçütler (i) en az 0,9 büzülme, (ii) görünen ve düzeltilmiş Nagelkerke R kare değerleri arasında en fazla 0,05 fark ve (iii) genel riskin 0,05 içinde kestirilmesidir."""
        r2_max = 1 - np.exp(2 * (prevalence * np.log(prevalence) + (1 - prevalence) * np.log(1 - prevalence)))
        n1 = P / ((shrinkage - 1) * np.log(1 - r2_cs / shrinkage))
        s2 = r2_cs / (r2_cs + delta * r2_max)
        n2 = P / ((s2 - 1) * np.log(1 - r2_cs / s2))
        n3 = (1.96 / margin) ** 2 * prevalence * (1 - prevalence)
        return {"(i)": math.ceil(n1), "(ii)": math.ceil(n2), "(iii)": math.ceil(n3), "R2_max": round(float(r2_max), 3)}

    REFERENCE = ["age", "sex", "cp_2.0", "cp_3.0", "cp_4.0"]
    anticipated = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[REFERENCE], y_train)
    r2_cs = cox_snell_r2(y_train, anticipated.predict_proba(X_train[REFERENCE])[:, 1])
    P = X_train.shape[1]
    sizes = riley_sizes(P, float(y_train.mean()), r2_cs)

    ALL = list(X.columns)


    def fit(columns):
        """Eğitim hastalarında bir lojistik regresyon kurar; ölçekleme işlem hattının içinde yer alır ve yalnızca eğitim hastalarından öğrenilir."""
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[columns], y_train)


    reference, model = fit(REFERENCE), fit(ALL)
    p_ref = reference.predict_proba(X_test[REFERENCE])[:, 1]
    p_model = model.predict_proba(X_test[ALL])[:, 1]
    yt = y_test.to_numpy()

    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    p_cv = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X_train[ALL], y_train, cv=folds, method="predict_proba")[:, 1]
    GRID = np.round(np.arange(0.01, 0.51, 0.01), 2)
    threshold = max(t for t in GRID if (p_cv[y_train.to_numpy() == 1] >= t).mean() >= 0.95)
    ruled_out = p_model < threshold
    counts = {"TP": int((~ruled_out & (yt == 1)).sum()), "FN": int((ruled_out & (yt == 1)).sum()),
              "TN": int((ruled_out & (yt == 0)).sum()), "FP": int((~ruled_out & (yt == 0)).sum())}
    p_ref_train = reference.predict_proba(X_train[REFERENCE])[:, 1]
    globals()["_course_cache"] = dict(heart=heart, X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test, p_ref=p_ref,
                                      p_model=p_model, p_cv=p_cv, p_ref_train=p_ref_train, threshold=threshold)
    return _course_cache


def _patients(which):
    """Eğitim ya da test hastaları veya ayırmanın sırasıyla ikisi birden; kümeleri, sekiz klinik değişken ve hastalıkla birlikte."""
    c = _course()
    parts = {"eğitim": c["X_train"].index, "test": c["X_test"].index}
    rows = [(s, i) for s, idx in parts.items() for i in idx if which == "both" or s == which]
    h = c["heart"].loc[[i for _, i in rows]]
    return pd.DataFrame({"hasta": h.index, "küme": [s for s, _ in rows],
                         "yaş": h["age"].astype(int), "cinsiyet": h["sex"].map({1.0: "erkek", 0.0: "kadın"}),
                         "göğüs ağrısı tipi": h["cp"].map(_CP),
                         "istirahat kan basıncı": h["trestbps"].astype(int), "kolesterol": h["chol"].astype(int),
                         "en yüksek kalp hızı": h["thalach"].astype(int), "egzersiz anjinası": h["exang"].astype(int),
                         "ST çökmesi": h["oldpeak"], "hastalık": h["disease"]}).reset_index(drop=True)


_c = _course()
print(len(_c["heart"]), "hasta Cleveland verisinde:", len(_c["X_train"]), "eğitim ve", len(_c["X_test"]), "test için")
print("2. durak hazır.")

### Adım 2.1: İki AUC değerinin aralıkları

**Amaç.** Bu adımda referansın ve tam modelin test AUC değerleri bootstrap aralıklarıyla birlikte verilir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Final projesinin raporu her sonucu belirsizliğiyle birlikte verir. 76 test hastasıyla kardiyolog, aynı türden başka hastalarla her AUC değerinin ne kadar değişebileceğini öğrenmek istiyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda UCI Heart Disease veri setinin Cleveland verisinden, yani bir egzersiz testinden sonra koroner anjiyografiye gönderilmiş hastalardan 76 test hastası bulunur. Her satır bir hastadır: "hasta" hastanın verideki numarasını, "olasılık, referans" dersin bir referans modelinin (yaş, cinsiyet ve göğüs ağrısı tipiyle kurulan bir lojistik regresyon) verdiği koroner arter hastalığı olasılığını, "olasılık, model" ise dersin tam modelinin (buna istirahat kan basıncı, kolesterol ve egzersiz testinin üç sonucu eklenmiş hâli) verdiği olasılığı gösterir. İki model de yalnızca 227 eğitim hastasında kurulmuştur. "hastalık", anjiyogramda en az bir ana koroner arterde çapın yüzde 50'sinden fazla daralma görüldüyse 1, aksi hâlde 0 olur.  
> Görev: Test hastalarından 2.000 yeniden örneklem çek: Her yeniden örneklem, 2026 seed değeriyle başlatılan bir rastgele sayı üreteciyle, tablodaki hasta sayısı kadar hastayı yerine koyarak çeker. Yalnızca tek bir sonucun hastalarını içeren yeniden örneklemi atla. Her yeniden örneklemde aynı hastalarda referansın ve modelin ROC eğrisi altındaki alanını (AUC) ve farklarını (model eksi referans) hesapla. Bir sonucun yüzde 95'lik aralığı, değerlerinin yeniden örneklemler boyunca 2,5. yüzdeliğinden 97,5. yüzdeliğine uzanır.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Kullanılan yeniden örneklem sayısını ve her modelin ve farkın bütün test hastalarındaki AUC değerini 2,5. ve 97,5. yüzdelikleriyle veren bir tabloyu yazdır; modelin AUC değerinin 2,5. yüzdeliğini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Her aralık bütün test hastalarındaki AUC değerini içermeli ve aynı yeniden örneklemler iki model için de kullanılmalı.  

In [ ]:
#@title Adım 2.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_c = _course()
veri = pd.DataFrame({"hasta": _c["X_test"].index, "hastalık": _c["y_test"].to_numpy(),
                     "olasılık, referans": np.round(_c["p_ref"], 6), "olasılık, model": np.round(_c["p_model"], 6)})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.1 kontrolü
teach.check_step("Adım 2.1", "cevap", globals())

In [ ]:
#@title Adım 2.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_c = _course()
veri = pd.DataFrame({"hasta": _c["X_test"].index, "hastalık": _c["y_test"].to_numpy(),
                     "olasılık, referans": np.round(_c["p_ref"], 6), "olasılık, model": np.round(_c["p_model"], 6)})
y = veri["hastalık"].to_numpy()
p_ref = veri["olasılık, referans"].to_numpy()
p_model = veri["olasılık, model"].to_numpy()
rng = np.random.default_rng(2026)
satirlar = []
for _ in range(2000):
    i = rng.integers(0, len(y), len(y))
    if y[i].min() == y[i].max():
        continue
    a, b = roc_auc_score(y[i], p_ref[i]), roc_auc_score(y[i], p_model[i])
    satirlar.append({"referans": a, "model": b, "fark": b - a})
boot = pd.DataFrame(satirlar)
tablo = pd.DataFrame({"AUC": [roc_auc_score(y, p_ref), roc_auc_score(y, p_model), roc_auc_score(y, p_model) - roc_auc_score(y, p_ref)],
                      "2.5 %": boot.quantile(0.025).to_numpy(), "97.5 %": boot.quantile(0.975).to_numpy()}, index=boot.columns).round(3)
print(len(boot), "yeniden örneklem")
print(tablo)
cevap = round(float(np.percentile(boot["model"], 2.5)), 3)
teach.check_step("Adım 2.1", "cevap", globals())

**Sonuç ne gösteriyor?** Tam modelin AUC değerinin alt sınırı 0,872 düzeyindedir: 76 hastayla her modelin aralığı AUC ölçeğinin yaklaşık onda birini kaplar.

### Adım 2.2: Kazancın aralığı

**Amaç.** Bu adımda tam model ile referans arasındaki fark, aynı yeniden örneklemlerde hesaplanan bootstrap aralığıyla birlikte verilir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Projenin sorusu, egzersiz testinin sonuçlarının kardiyoloğun testten önce bildiklerine bir şey katıp katmadığıdır. Cevap yan yana konan iki AUC değeri değil, farkın aralığıdır.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda UCI Heart Disease veri setinin Cleveland verisinden, yani bir egzersiz testinden sonra koroner anjiyografiye gönderilmiş hastalardan 76 test hastası bulunur. Her satır bir hastadır: "hasta" hastanın verideki numarasını, "olasılık, referans" dersin bir referans modelinin (yaş, cinsiyet ve göğüs ağrısı tipiyle kurulan bir lojistik regresyon) verdiği koroner arter hastalığı olasılığını, "olasılık, model" ise dersin tam modelinin (buna istirahat kan basıncı, kolesterol ve egzersiz testinin üç sonucu eklenmiş hâli) verdiği olasılığı gösterir. İki model de yalnızca 227 eğitim hastasında kurulmuştur. "hastalık", anjiyogramda en az bir ana koroner arterde çapın yüzde 50'sinden fazla daralma görüldüyse 1, aksi hâlde 0 olur.  
> Görev: Test hastalarından 2.000 yeniden örneklem çek: Her yeniden örneklem, 2026 seed değeriyle başlatılan bir rastgele sayı üreteciyle, tablodaki hasta sayısı kadar hastayı yerine koyarak çeker. Yalnızca tek bir sonucun hastalarını içeren yeniden örneklemi atla. Her yeniden örneklemde aynı hastalarda referansın ve modelin ROC eğrisi altındaki alanını (AUC) ve farklarını (model eksi referans) hesapla. Bir sonucun yüzde 95'lik aralığı, değerlerinin yeniden örneklemler boyunca 2,5. yüzdeliğinden 97,5. yüzdeliğine uzanır. Ayrıca farkın 0 ya da daha küçük olduğu yeniden örneklemlerin payını ver.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Üç sonucun aralıklarıyla birlikte tablosunu ve modelin daha iyi olmadığı yeniden örneklemlerin payını yazdır; farkın 2,5. yüzdeliğini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Fark her yeniden örneklemde o yeniden örneklemin iki AUC değerinden hesaplanmalı; iki ayrı yeniden örneklem kümesinden hesaplanmamalı.  

In [ ]:
#@title Adım 2.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_c = _course()
veri = pd.DataFrame({"hasta": _c["X_test"].index, "hastalık": _c["y_test"].to_numpy(),
                     "olasılık, referans": np.round(_c["p_ref"], 6), "olasılık, model": np.round(_c["p_model"], 6)})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.2 kontrolü
teach.check_step("Adım 2.2", "cevap", globals())

In [ ]:
#@title Adım 2.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_c = _course()
veri = pd.DataFrame({"hasta": _c["X_test"].index, "hastalık": _c["y_test"].to_numpy(),
                     "olasılık, referans": np.round(_c["p_ref"], 6), "olasılık, model": np.round(_c["p_model"], 6)})
y = veri["hastalık"].to_numpy()
p_ref = veri["olasılık, referans"].to_numpy()
p_model = veri["olasılık, model"].to_numpy()
rng = np.random.default_rng(2026)
satirlar = []
for _ in range(2000):
    i = rng.integers(0, len(y), len(y))
    if y[i].min() == y[i].max():
        continue
    a, b = roc_auc_score(y[i], p_ref[i]), roc_auc_score(y[i], p_model[i])
    satirlar.append({"referans": a, "model": b, "fark": b - a})
boot = pd.DataFrame(satirlar)
tablo = pd.DataFrame({"AUC": [roc_auc_score(y, p_ref), roc_auc_score(y, p_model), roc_auc_score(y, p_model) - roc_auc_score(y, p_ref)],
                      "2.5 %": boot.quantile(0.025).to_numpy(), "97.5 %": boot.quantile(0.975).to_numpy()}, index=boot.columns).round(3)
print(len(boot), "yeniden örneklem")
print(tablo)
print("modelin daha iyi olmadığı yeniden örneklemlerin payı:", round(float((boot["fark"] <= 0).mean()), 3))
cevap = round(float(np.percentile(boot["fark"], 2.5)), 3)
teach.check_step("Adım 2.2", "cevap", globals())

**Sonuç ne gösteriyor?** Farkın aralığı sıfırın altında, −0,020 değerinden başlar: Bu verilerde tam model referanstan açıkça daha iyi değildir. Rapor bunu aralığıyla birlikte söyler.

### Adım 2.3: Test hastalarında kalibrasyon

**Amaç.** Bu adımda tam modelin test hastalarındaki kalibrasyon sabiti ve eğimi hesaplanır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Bir dışlama olasılıkların kendisini kullanır. Kardiyolog, olasılıkların ortalamada doğru olup olmadığını ve fazla uç ya da fazla ılımlı olup olmadıklarını öğrenmek istiyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda UCI Heart Disease veri setinin Cleveland verisinden, yani bir egzersiz testinden sonra koroner anjiyografiye gönderilmiş hastalardan 76 test hastası bulunur. Her satır bir hastadır: "hasta" hastanın verideki numarasını, "olasılık, model" ise dersin sekiz klinik değişkenle ve yalnızca 227 eğitim hastasında kurulan bir lojistik regresyonunun verdiği koroner arter hastalığı olasılığını gösterir. "hastalık", anjiyogramda en az bir ana koroner arterde çapın yüzde 50'sinden fazla daralma görüldüyse 1, aksi hâlde 0 olur.  
> Görev: Her olasılığı log-odds değerine, yani olasılığın bir eksi olasılığa bölümünün doğal logaritmasına dönüştür. Kalibrasyon sabiti, hastalık için yalnızca bir sabit terim içeren ve log-odds değerini sabit bir kayma olarak alan bir lojistik regresyondan gelir. Kalibrasyon eğimi ise hastalık için bir sabit terim ve tek girdi olarak log-odds içeren bir lojistik regresyonda log-odds değerinin katsayısıdır. İkisini de hiçbir ceza terimi olmadan, örneğin Newton adımlarıyla kur.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Ortalama olasılığı, gözlenen hastalık oranını, kalibrasyon sabitini ve kalibrasyon eğimini yazdır; eğimi üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Kusursuz kalibre edilmiş bir model 0 sabit ve 1 eğim verirdi; sabitin işareti, gözlenen oran eksi ortalama olasılık farkının işaretiyle aynı olmalı.  

In [ ]:
#@title Adım 2.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_c = _course()
veri = pd.DataFrame({"hasta": _c["X_test"].index, "hastalık": _c["y_test"].to_numpy(),
                     "olasılık, model": np.round(_c["p_model"], 6)})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.3 kontrolü
teach.check_step("Adım 2.3", "cevap", globals())

In [ ]:
#@title Adım 2.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_c = _course()
veri = pd.DataFrame({"hasta": _c["X_test"].index, "hastalık": _c["y_test"].to_numpy(),
                     "olasılık, model": np.round(_c["p_model"], 6)})
y = veri["hastalık"].to_numpy(dtype=float)
p = veri["olasılık, model"].to_numpy(dtype=float)
lp = np.log(p / (1 - p))
a = 0.0
for _ in range(50):
    q = 1 / (1 + np.exp(-(a + lp)))
    a += (y - q).sum() / (q * (1 - q)).sum()
Z = np.c_[np.ones_like(lp), lp]
b = np.zeros(2)
for _ in range(50):
    q = 1 / (1 + np.exp(-Z @ b))
    b += np.linalg.solve(Z.T @ (Z * (q * (1 - q))[:, None]), Z.T @ (y - q))
print("ortalama olasılık", round(float(p.mean()), 3), "| gözlenen oran", round(float(y.mean()), 3))
print("kalibrasyon sabiti", round(float(a), 3), "| kalibrasyon eğimi", round(float(b[1]), 3))
cevap = round(float(b[1]), 3)
teach.check_step("Adım 2.3", "cevap", globals())

**Sonuç ne gösteriyor?** Eğim 76 hastada 1,241 olarak bulunur ve aralığı 1'i içerir. Eğitim hastalarının çapraz doğrulama olasılıkları 1'in biraz altında bir eğim verir; bu da (i) ölçütünün beklediği hafif aşırı öğrenmedir.

**2. durak tamamlandı.** [Ders sayfasına](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-11/ders.html#stop-2) dönün ve duraktan sonrasını okumaya devam edin.

## Durak 3: Dışlama ve sınırları

*Ders: [3. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-11/ders.html#stop-3)* · *Python yolunda aynı sonuç: Kısım 2, 7-10. bölümler*

Bu durak olasılıkları bir karara dönüştürür. Dışlama eşiği eğitim hastalarının çapraz doğrulama olasılıklarında seçilir; test hastaları bu eşiğin kaç sevki önleyeceğini, test kümesindeki kadınlar ve erkekler de dışlamanın iki grupta da güvenli sayılıp sayılamayacağını gösterir.

**Durağın sonunda:** Dışlama eşiğini (0,18), dışlanan test hastalarını (27) ve kadınlarda duyarlılığın alt sınırını (0,646) bulacaksınız.

In [ ]:
#@title 3. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
_CP = {1.0: "tipik anjina", 2.0: "atipik anjina", 3.0: "anjina dışı ağrı", 4.0: "ağrı yok"}


def _course():
    """Haftanın çözümlü projesi: Cleveland verisi, 2. haftanın ayırması, referans, tam model ve eğitim hastalarının çapraz doğrulama olasılıkları (bir kez hesaplanır, birkaç saniye sürer)."""
    if "_course_cache" in globals():
        return _course_cache
    heart = load_cleveland()
    FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
    missing = heart.isna().sum()
    audit = pd.DataFrame({"eksik değer": missing, "projede kullanılıyor": [c in FEATURES for c in missing.index]})

    cad = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"], drop_first=True, dtype=int)
    X, y = cad.drop(columns="disease"), cad["disease"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)


    def cox_snell_r2(y, p):
        """Tahmin edilen p olasılıklarının y sonuçları için Cox-Snell R kare değerini hesaplar."""
        y, p = np.asarray(y, dtype=float), np.clip(np.asarray(p, dtype=float), 1e-12, 1 - 1e-12)
        phi = y.mean()
        ll_null = len(y) * (phi * np.log(phi) + (1 - phi) * np.log(1 - phi))
        ll_model = np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
        return 1 - np.exp(2 * (ll_null - ll_model) / len(y))


    def riley_sizes(P, prevalence, r2_cs, shrinkage=0.9, delta=0.05, margin=0.05):
        """Riley ve arkadaşlarının P parametreli bir ikili sonuç modeli için önerdiği üç büyüklüğü hesaplar. Ölçütler (i) en az 0,9 büzülme, (ii) görünen ve düzeltilmiş Nagelkerke R kare değerleri arasında en fazla 0,05 fark ve (iii) genel riskin 0,05 içinde kestirilmesidir."""
        r2_max = 1 - np.exp(2 * (prevalence * np.log(prevalence) + (1 - prevalence) * np.log(1 - prevalence)))
        n1 = P / ((shrinkage - 1) * np.log(1 - r2_cs / shrinkage))
        s2 = r2_cs / (r2_cs + delta * r2_max)
        n2 = P / ((s2 - 1) * np.log(1 - r2_cs / s2))
        n3 = (1.96 / margin) ** 2 * prevalence * (1 - prevalence)
        return {"(i)": math.ceil(n1), "(ii)": math.ceil(n2), "(iii)": math.ceil(n3), "R2_max": round(float(r2_max), 3)}

    REFERENCE = ["age", "sex", "cp_2.0", "cp_3.0", "cp_4.0"]
    anticipated = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[REFERENCE], y_train)
    r2_cs = cox_snell_r2(y_train, anticipated.predict_proba(X_train[REFERENCE])[:, 1])
    P = X_train.shape[1]
    sizes = riley_sizes(P, float(y_train.mean()), r2_cs)

    ALL = list(X.columns)


    def fit(columns):
        """Eğitim hastalarında bir lojistik regresyon kurar; ölçekleme işlem hattının içinde yer alır ve yalnızca eğitim hastalarından öğrenilir."""
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[columns], y_train)


    reference, model = fit(REFERENCE), fit(ALL)
    p_ref = reference.predict_proba(X_test[REFERENCE])[:, 1]
    p_model = model.predict_proba(X_test[ALL])[:, 1]
    yt = y_test.to_numpy()

    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    p_cv = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X_train[ALL], y_train, cv=folds, method="predict_proba")[:, 1]
    GRID = np.round(np.arange(0.01, 0.51, 0.01), 2)
    threshold = max(t for t in GRID if (p_cv[y_train.to_numpy() == 1] >= t).mean() >= 0.95)
    ruled_out = p_model < threshold
    counts = {"TP": int((~ruled_out & (yt == 1)).sum()), "FN": int((ruled_out & (yt == 1)).sum()),
              "TN": int((ruled_out & (yt == 0)).sum()), "FP": int((~ruled_out & (yt == 0)).sum())}
    p_ref_train = reference.predict_proba(X_train[REFERENCE])[:, 1]
    globals()["_course_cache"] = dict(heart=heart, X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test, p_ref=p_ref,
                                      p_model=p_model, p_cv=p_cv, p_ref_train=p_ref_train, threshold=threshold)
    return _course_cache


def _patients(which):
    """Eğitim ya da test hastaları veya ayırmanın sırasıyla ikisi birden; kümeleri, sekiz klinik değişken ve hastalıkla birlikte."""
    c = _course()
    parts = {"eğitim": c["X_train"].index, "test": c["X_test"].index}
    rows = [(s, i) for s, idx in parts.items() for i in idx if which == "both" or s == which]
    h = c["heart"].loc[[i for _, i in rows]]
    return pd.DataFrame({"hasta": h.index, "küme": [s for s, _ in rows],
                         "yaş": h["age"].astype(int), "cinsiyet": h["sex"].map({1.0: "erkek", 0.0: "kadın"}),
                         "göğüs ağrısı tipi": h["cp"].map(_CP),
                         "istirahat kan basıncı": h["trestbps"].astype(int), "kolesterol": h["chol"].astype(int),
                         "en yüksek kalp hızı": h["thalach"].astype(int), "egzersiz anjinası": h["exang"].astype(int),
                         "ST çökmesi": h["oldpeak"], "hastalık": h["disease"]}).reset_index(drop=True)


_c = _course()
print(len(_c["heart"]), "hasta Cleveland verisinde:", len(_c["X_train"]), "eğitim ve", len(_c["X_test"]), "test için")
print("3. durak hazır.")

### Adım 3.1: Dışlama eşiği

**Amaç.** Bu adımda dışlama eşiği eğitim hastalarının çapraz doğrulama olasılıklarında seçilir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Eşiğin altındaki bir hasta anjiyografiye gönderilmezdi; bu yüzden eşik çok yüksek bir duyarlılığı korumalıdır: Hastalığı olanların en az yüzde 95'i eşikte ya da eşiğin üstünde kalmalıdır. Test hastaları bu seçimde hiçbir rol oynamamalıdır.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda UCI Heart Disease veri setinin Cleveland verisinden, yani bir egzersiz testinden sonra koroner anjiyografiye gönderilmiş hastalardan 227 eğitim hastası bulunur. Her satır bir hastadır: "hasta" hastanın verideki numarasını, "olasılık, çapraz doğrulama" ise dersin sekiz klinik değişkenle kurulan bir lojistik regresyonunun beş katlı çapraz doğrulamayla hesaplanan koroner arter hastalığı olasılığını verir: Her hasta, eğitim hastalarının öteki beşte dördünde kurulmuş bir modelin olasılığını alır. "hastalık", anjiyogramda en az bir ana koroner arterde çapın yüzde 50'sinden fazla daralma görüldüyse 1, aksi hâlde 0 olur.  
> Görev: 0,01, 0,02 ve bu biçimde 0,50'ye kadar giden eşikler arasından, hastalığı olan eğitim hastalarının en az yüzde 95'inin olasılığının ona eşit ya da ondan büyük olduğu en yüksek eşiği al. Bu eşikteki duyarlılığı ve olasılığı eşiğin altında kalan eğitim hastalarının payını ver.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Eşiği, eğitim hastalarındaki duyarlılığı ve dışlanan payı yazdır; eşiği iki ondalık basamakla "cevap" adıyla sakla.  
> Test: Eşikteki duyarlılık en az 0,95 olmalı; ızgaradaki bir sonraki eşikte ise 0,95'in altına düşmeli.  

In [ ]:
#@title Adım 3.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_c = _course()
veri = pd.DataFrame({"hasta": _c["X_train"].index, "hastalık": _c["y_train"].to_numpy(),
                     "olasılık, çapraz doğrulama": np.round(_c["p_cv"], 6)})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.1 kontrolü
teach.check_step("Adım 3.1", "cevap", globals())

In [ ]:
#@title Adım 3.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_c = _course()
veri = pd.DataFrame({"hasta": _c["X_train"].index, "hastalık": _c["y_train"].to_numpy(),
                     "olasılık, çapraz doğrulama": np.round(_c["p_cv"], 6)})
y = veri["hastalık"].to_numpy()
p = veri["olasılık, çapraz doğrulama"].to_numpy()
izgara = np.round(np.arange(0.01, 0.51, 0.01), 2)
esik = max(t for t in izgara if (p[y == 1] >= t).mean() >= 0.95)
print("eşik", esik, "| duyarlılık, eğitim", round(float((p[y == 1] >= esik).mean()), 3),
      "| dışlanan pay, eğitim", round(float((p < esik).mean()), 3))
cevap = float(esik)
teach.check_step("Adım 3.1", "cevap", globals())

**Sonuç ne gösteriyor?** Eşik 0,18 olarak bulunur. Tek bir test hastası kullanılmadan seçildiği için bir sonraki adımın test sonucu yeni hastalar için dürüst bir tahmindir.

### Adım 3.2: Test hastalarında dışlama

**Amaç.** Bu adımda eşik test hastalarına bir kez uygulanır ve duyarlılık, özgüllük ve negatif öngörü değeri Wilson aralıklarıyla birlikte verilir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kardiyolog, dışlamanın kaç sevki önleyeceğini, hastalığı olan herhangi bir hastayı kaçırıp kaçırmadığını ve 76 hastayla bu sayılardan ne kadar emin olunabileceğini öğrenmek istiyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda UCI Heart Disease veri setinin Cleveland verisinden, yani bir egzersiz testinden sonra koroner anjiyografiye gönderilmiş hastalardan 76 test hastası bulunur. Her satır bir hastadır: "hasta" hastanın verideki numarasını, "olasılık, model" ise dersin sekiz klinik değişkenle ve yalnızca 227 eğitim hastasında kurulan bir lojistik regresyonunun verdiği koroner arter hastalığı olasılığını gösterir. "hastalık", anjiyogramda en az bir ana koroner arterde çapın yüzde 50'sinden fazla daralma görüldüyse 1, aksi hâlde 0 olur. Projenin eğitim hastalarında seçilen dışlama eşiği 0,18 değerindedir: Olasılığı bu eşiğin altında kalan bir hasta anjiyografiye gönderilmezdi.  
> Görev: Olasılığı eşiğin altında kalan hasta dışlanır. Dışlananlar ve sevk edilenler arasında hastalığı olan ve olmayan hastaları say; duyarlılığı, özgüllüğü ve negatif öngörü değerini, yani dışlanan hastalar arasında hastalığı olmayanların payını hesapla. Her oranı yüzde 95'lik Wilson aralığıyla ver: n hastanın k tanesi için, z = 1,96 olmak üzere merkez (k/n artı z kare bölü 2n) bölü (1 artı z kare bölü n), yarı genişlik ise z çarpı (k/n çarpı bir eksi k/n, bölü n, artı z kare bölü 4n kare) değerinin karekökü, bölü (1 artı z kare bölü n) olur.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Dört sayının tablosunu göster, üç oranı Wilson aralıklarıyla birlikte yazdır; dışlanan hasta sayısını "cevap" adıyla sakla.  
> Test: Dört sayının toplamı 76 olmalı ve her aralık 0 ile 1 arasında kalıp kendi oranını içermeli.  

In [ ]:
#@title Adım 3.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_c = _course()
veri = pd.DataFrame({"hasta": _c["X_test"].index, "hastalık": _c["y_test"].to_numpy(),
                     "olasılık, model": np.round(_c["p_model"], 6)})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.2 kontrolü
teach.check_step("Adım 3.2", "cevap", globals())

In [ ]:
#@title Adım 3.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_c = _course()
veri = pd.DataFrame({"hasta": _c["X_test"].index, "hastalık": _c["y_test"].to_numpy(),
                     "olasılık, model": np.round(_c["p_model"], 6)})


def wilson(k, n, z=1.96):
    c = (k / n + z ** 2 / (2 * n)) / (1 + z ** 2 / n)
    h = z * np.sqrt(k / n * (1 - k / n) / n + z ** 2 / (4 * n ** 2)) / (1 + z ** 2 / n)
    return round(float(c - h), 3), round(float(c + h), 3)


y = veri["hastalık"].to_numpy()
disarida = veri["olasılık, model"].to_numpy() < 0.18
DP, YN = int((~disarida & (y == 1)).sum()), int((disarida & (y == 1)).sum())
DN, YP = int((disarida & (y == 0)).sum()), int((~disarida & (y == 0)).sum())
print(pd.DataFrame({"hastalık": [DP, YN], "hastalık yok": [YP, DN]}, index=["sevk edilir", "dışlanır"]))
print("duyarlılık", round(DP / (DP + YN), 3), wilson(DP, DP + YN))
print("özgüllük", round(DN / (DN + YP), 3), wilson(DN, DN + YP))
print("negatif öngörü değeri", round(DN / (DN + YN), 3), wilson(DN, DN + YN))
cevap = int(disarida.sum())
teach.check_step("Adım 3.2", "cevap", globals())

**Sonuç ne gösteriyor?** Eşik, hastalığı olan hiçbir hastayı kaçırmadan 76 test hastasının 27 tanesini dışlar. Negatif öngörü değerinin alt sınırı ise bir dışlamanın ihtiyaç duyduğu değerden hâlâ uzaktır; bir sonraki aşamanın daha büyük bir doğrulama olmasının nedeni budur.

### Adım 3.3: Kadınlar ve erkekler

**Amaç.** Bu adımda dışlamanın kadınlardaki ve erkeklerdeki duyarlılığı Wilson aralıklarının alt sınırlarıyla birlikte verilir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Bir dışlama kullanıldığı her grupta güvenli olmalıdır. Test kümesinde kadınlar erkeklerden azdır ve kadınların daha azında hastalık vardır.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda UCI Heart Disease veri setinin Cleveland verisinden, yani bir egzersiz testinden sonra koroner anjiyografiye gönderilmiş hastalardan 76 test hastası bulunur. Her satır bir hastadır: "hasta" hastanın verideki numarasını, "cinsiyet" "erkek" ya da "kadın" değerini, "olasılık, model" ise dersin sekiz klinik değişkenle ve yalnızca 227 eğitim hastasında kurulan bir lojistik regresyonunun verdiği koroner arter hastalığı olasılığını gösterir. "hastalık", anjiyogramda en az bir ana koroner arterde çapın yüzde 50'sinden fazla daralma görüldüyse 1, aksi hâlde 0 olur. Projenin eğitim hastalarında seçilen dışlama eşiği 0,18 değerindedir.  
> Görev: Kadınlar ve erkekler için ayrı ayrı hastaları, hastalığı olanları, hastalığı olup eşikte ya da eşiğin üstünde kalanları ve dışlananları say; duyarlılığı hesapla. Her oranı yüzde 95'lik Wilson aralığıyla ver: n hastanın k tanesi için, z = 1,96 olmak üzere merkez (k/n artı z kare bölü 2n) bölü (1 artı z kare bölü n), yarı genişlik ise z çarpı (k/n çarpı bir eksi k/n, bölü n, artı z kare bölü 4n kare) değerinin karekökü, bölü (1 artı z kare bölü n) olur.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her cinsiyet için bir satırdan oluşan ve duyarlılığın Wilson aralığının alt sınırını içeren bir tablo göster; kadınlar için alt sınırı üç ondalık basamakla "cevap" adıyla sakla.  
> Test: İki satırın hastalarının toplamı 76 olmalı ve 1 olan bir duyarlılığın üst sınırı 1 olmalı.  

In [ ]:
#@title Adım 3.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_c = _course()
veri = pd.DataFrame({"hasta": _c["X_test"].index, "cinsiyet": _c["X_test"]["sex"].map({1: "erkek", 0: "kadın"}).to_numpy(),
                     "hastalık": _c["y_test"].to_numpy(), "olasılık, model": np.round(_c["p_model"], 6)})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.3 kontrolü
teach.check_step("Adım 3.3", "cevap", globals())

In [ ]:
#@title Adım 3.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_c = _course()
veri = pd.DataFrame({"hasta": _c["X_test"].index, "cinsiyet": _c["X_test"]["sex"].map({1: "erkek", 0: "kadın"}).to_numpy(),
                     "hastalık": _c["y_test"].to_numpy(), "olasılık, model": np.round(_c["p_model"], 6)})


def wilson(k, n, z=1.96):
    c = (k / n + z ** 2 / (2 * n)) / (1 + z ** 2 / n)
    h = z * np.sqrt(k / n * (1 - k / n) / n + z ** 2 / (4 * n ** 2)) / (1 + z ** 2 / n)
    return round(float(c - h), 3), round(float(c + h), 3)


d = veri
disarida = d["olasılık, model"].to_numpy() < 0.18
y = d["hastalık"].to_numpy()
satirlar = []
for grup in ["kadın", "erkek"]:
    g = (d["cinsiyet"] == grup).to_numpy()
    k, n = int((~disarida & (y == 1) & g).sum()), int(((y == 1) & g).sum())
    satirlar.append({"grup": grup, "hasta": int(g.sum()), "hastalıklı": n,
                         "duyarlılık": round(k / n, 3), "alt sınır": wilson(k, n)[0],
                         "dışlanan": int((disarida & g).sum())})
tablo = pd.DataFrame(satirlar)
print(tablo.to_string(index=False))
cevap = float(tablo.iloc[0]["alt sınır"])
teach.check_step("Adım 3.3", "cevap", globals())

**Sonuç ne gösteriyor?** Hastalığı olan hiçbir kadın kaçırılmaz; buna rağmen kadınlarda duyarlılığın alt sınırı yalnızca 0,646 düzeyindedir: Hastalığı olan yedi kadın, dışlamanın kadınlar için güvenli olduğunu gösteremez. Rapor bu sınırı adlandırır ve doğrulamanın planı yeterli sayıda kadın hasta ister.

**3. durak tamamlandı.** [Ders sayfasına](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-11/ders.html#stop-3) dönün ve duraktan sonrasını okumaya devam edin.

## Prompt yolunda haftalık proje

*Haftanın projesi: [11. haftanın README sayfası, Haftalık proje](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-11/README.md#haftal%C4%B1k-proje)*

Haftalık proje, her adım için bir prompt olmak üzere aşağıdaki promptlarla da yapılabilir. Önce hazırlık hücresini bir kez çalıştırın; hücre projenin verilerini yükler. Ardından her adım, duraklardaki adımlar gibi ilerler. Tek fark şudur: Projede kontrol hücresi ve hazır kod yoktur, çünkü sonuçlar rapora girer. Bir promptun testi sağlanmadan sonraki prompta geçilmez ve her prompt YZ kullanım günlüğüne yazılır.

In [ ]:
#@title Projenin hazırlığı: Bu hücreyi bir kez çalıştırın (projenin verilerini yükler)
_CP = {1.0: "tipik anjina", 2.0: "atipik anjina", 3.0: "anjina dışı ağrı", 4.0: "ağrı yok"}


def _course():
    """Haftanın çözümlü projesi: Cleveland verisi, 2. haftanın ayırması, referans, tam model ve eğitim hastalarının çapraz doğrulama olasılıkları (bir kez hesaplanır, birkaç saniye sürer)."""
    if "_course_cache" in globals():
        return _course_cache
    heart = load_cleveland()
    FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
    missing = heart.isna().sum()
    audit = pd.DataFrame({"eksik değer": missing, "projede kullanılıyor": [c in FEATURES for c in missing.index]})

    cad = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"], drop_first=True, dtype=int)
    X, y = cad.drop(columns="disease"), cad["disease"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)


    def cox_snell_r2(y, p):
        """Tahmin edilen p olasılıklarının y sonuçları için Cox-Snell R kare değerini hesaplar."""
        y, p = np.asarray(y, dtype=float), np.clip(np.asarray(p, dtype=float), 1e-12, 1 - 1e-12)
        phi = y.mean()
        ll_null = len(y) * (phi * np.log(phi) + (1 - phi) * np.log(1 - phi))
        ll_model = np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
        return 1 - np.exp(2 * (ll_null - ll_model) / len(y))


    def riley_sizes(P, prevalence, r2_cs, shrinkage=0.9, delta=0.05, margin=0.05):
        """Riley ve arkadaşlarının P parametreli bir ikili sonuç modeli için önerdiği üç büyüklüğü hesaplar. Ölçütler (i) en az 0,9 büzülme, (ii) görünen ve düzeltilmiş Nagelkerke R kare değerleri arasında en fazla 0,05 fark ve (iii) genel riskin 0,05 içinde kestirilmesidir."""
        r2_max = 1 - np.exp(2 * (prevalence * np.log(prevalence) + (1 - prevalence) * np.log(1 - prevalence)))
        n1 = P / ((shrinkage - 1) * np.log(1 - r2_cs / shrinkage))
        s2 = r2_cs / (r2_cs + delta * r2_max)
        n2 = P / ((s2 - 1) * np.log(1 - r2_cs / s2))
        n3 = (1.96 / margin) ** 2 * prevalence * (1 - prevalence)
        return {"(i)": math.ceil(n1), "(ii)": math.ceil(n2), "(iii)": math.ceil(n3), "R2_max": round(float(r2_max), 3)}

    REFERENCE = ["age", "sex", "cp_2.0", "cp_3.0", "cp_4.0"]
    anticipated = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[REFERENCE], y_train)
    r2_cs = cox_snell_r2(y_train, anticipated.predict_proba(X_train[REFERENCE])[:, 1])
    P = X_train.shape[1]
    sizes = riley_sizes(P, float(y_train.mean()), r2_cs)

    ALL = list(X.columns)


    def fit(columns):
        """Eğitim hastalarında bir lojistik regresyon kurar; ölçekleme işlem hattının içinde yer alır ve yalnızca eğitim hastalarından öğrenilir."""
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[columns], y_train)


    reference, model = fit(REFERENCE), fit(ALL)
    p_ref = reference.predict_proba(X_test[REFERENCE])[:, 1]
    p_model = model.predict_proba(X_test[ALL])[:, 1]
    yt = y_test.to_numpy()

    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    p_cv = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X_train[ALL], y_train, cv=folds, method="predict_proba")[:, 1]
    GRID = np.round(np.arange(0.01, 0.51, 0.01), 2)
    threshold = max(t for t in GRID if (p_cv[y_train.to_numpy() == 1] >= t).mean() >= 0.95)
    ruled_out = p_model < threshold
    counts = {"TP": int((~ruled_out & (yt == 1)).sum()), "FN": int((ruled_out & (yt == 1)).sum()),
              "TN": int((ruled_out & (yt == 0)).sum()), "FP": int((~ruled_out & (yt == 0)).sum())}
    p_ref_train = reference.predict_proba(X_train[REFERENCE])[:, 1]
    globals()["_course_cache"] = dict(heart=heart, X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test, p_ref=p_ref,
                                      p_model=p_model, p_cv=p_cv, p_ref_train=p_ref_train, threshold=threshold)
    return _course_cache


def _patients(which):
    """Eğitim ya da test hastaları veya ayırmanın sırasıyla ikisi birden; kümeleri, sekiz klinik değişken ve hastalıkla birlikte."""
    c = _course()
    parts = {"eğitim": c["X_train"].index, "test": c["X_test"].index}
    rows = [(s, i) for s, idx in parts.items() for i in idx if which == "both" or s == which]
    h = c["heart"].loc[[i for _, i in rows]]
    return pd.DataFrame({"hasta": h.index, "küme": [s for s, _ in rows],
                         "yaş": h["age"].astype(int), "cinsiyet": h["sex"].map({1.0: "erkek", 0.0: "kadın"}),
                         "göğüs ağrısı tipi": h["cp"].map(_CP),
                         "istirahat kan basıncı": h["trestbps"].astype(int), "kolesterol": h["chol"].astype(int),
                         "en yüksek kalp hızı": h["thalach"].astype(int), "egzersiz anjinası": h["exang"].astype(int),
                         "ST çökmesi": h["oldpeak"], "hastalık": h["disease"]}).reset_index(drop=True)
print("Proje hazır.")

### Proje adımı 1: Verilerin denetimi ve ayırma

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Bir kardiyolog, egzersiz testinden sonra koroner arter hastalığını dışlamayı amaçlayan bir final projesi kuruyor. Herhangi bir modelden önce veriler denetlenir ve bir kez eğitim ve test hastalarına ayrılır. Kendi final projeniz için Veri satırını kendi tablonuza göre yeniden yazın: Sütunları, birimlerini ve sonucun anlamını kelimelerle anlatın; hiçbir zaman bir hastanın satırını eklemeyin.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda UCI Heart Disease veri setinin Cleveland verisinden, yani bir egzersiz testinden sonra koroner anjiyografiye gönderilmiş hastalardan 303 hasta, bütün sütunlarıyla birlikte bulunur. Her satır bir hastadır: "hasta" hastanın verideki numarasını (0-302) verir. Projenin sekiz değişkeni şunlardır: "yaş" (yıl), "cinsiyet" ("erkek" ya da "kadın"), "göğüs ağrısı tipi" ("tipik anjina", "atipik anjina", "anjina dışı ağrı" ya da "ağrı yok"), "istirahat kan basıncı" (mmHg), "kolesterol" (mg/dL), "en yüksek kalp hızı" (egzersiz testinde ulaşılan en yüksek kalp hızı, dakikada), "egzersiz anjinası" (egzersiz testi anjinaya yol açtıysa 1) ve "ST çökmesi" (egzersizde ST segmentinin istirahate göre çökmesi, mm). Veri setinden beş sütun daha gelir: "açlık kan şekeri 120 üstü" (açlık kan şekeri 120 mg/dL'nin üstündeyse 1), "istirahat EKG'si" (0 normal, 1 ST segmenti ya da T dalgası bozukluğu, 2 sol ventrikül hipertrofisi), "ST eğimi" (egzersizin doruğunda ST segmentinin eğimi: 1 yukarı, 2 düz, 3 aşağı), "floroskopide damar" (floroskopiyle boyanan ana damar sayısı, 0-3) ve "talyum sintigrafisi" (3 normal, 6 sabit defekt, 7 geri dönüşlü defekt). Boş bir hücre eksik bir değerdir. "hastalık", anjiyogramda en az bir ana koroner arterde çapın yüzde 50'sinden fazla daralma görüldüyse 1, aksi hâlde 0 olur.  
> Görev: Hasta sayısını, hastalığı olanların sayısını ve oranını ve her sütunun eksik değerlerini ver. Ardından hastaları, bir hastanın bütün satırları aynı kümede kalacak biçimde bir kez ayır: Dörtte biri test için ayrılsın, iki kümede hastalık oranı aynı olsun ve rastgele sayılar için 2026 seed değeri kullanılsın. Her kümedeki hasta sayısını ve hastalığı olan hasta sayısını ver.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Eksik değerlerin tablosunu ve iki kümenin tablosunu göster; hastalığı olan test hastalarının sayısını "cevap" adıyla sakla.  
> Test: İki küme bütün hastaları kapsamalı ve hiçbir hasta iki kümede birden bulunmamalı.  

In [ ]:
#@title Proje adımı 1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_h = _course()["heart"]
veri = pd.DataFrame({"hasta": _h.index, "yaş": _h["age"], "cinsiyet": _h["sex"].map({1.0: "erkek", 0.0: "kadın"}),
                     "göğüs ağrısı tipi": _h["cp"].map(_CP), "istirahat kan basıncı": _h["trestbps"],
                     "kolesterol": _h["chol"], "açlık kan şekeri 120 üstü": _h["fbs"],
                     "istirahat EKG'si": _h["restecg"], "en yüksek kalp hızı": _h["thalach"],
                     "egzersiz anjinası": _h["exang"], "ST çökmesi": _h["oldpeak"], "ST eğimi": _h["slope"],
                     "floroskopide damar": _h["ca"], "talyum sintigrafisi": _h["thal"],
                     "hastalık": _h["disease"]}).reset_index(drop=True)
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 2: Bir referans, model ve kazancın aralığı

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kardiyolog tam modeli, egzersiz testinden önce bilinenlerle aynı test hastalarında karşılaştırıyor ve kazancı aralığıyla birlikte veriyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda UCI Heart Disease veri setinin Cleveland verisinden, yani bir egzersiz testinden sonra koroner anjiyografiye gönderilmiş hastalardan 303 hasta bulunur. Her satır bir hastadır: "hasta" hastanın verideki numarasını (0-302), "küme" 2. haftanın kümesini verir ("eğitim", 227 hasta; "test", 76 hasta). Klinik değişkenler şunlardır: "yaş" (yıl), "cinsiyet" ("erkek" ya da "kadın"), "göğüs ağrısı tipi" ("tipik anjina", "atipik anjina", "anjina dışı ağrı" ya da "ağrı yok"), "istirahat kan basıncı" (mmHg), "kolesterol" (mg/dL), "en yüksek kalp hızı" (egzersiz testinde ulaşılan en yüksek kalp hızı, dakikada), "egzersiz anjinası" (egzersiz testi anjinaya yol açtıysa 1) ve "ST çökmesi" (egzersizde ST segmentinin istirahate göre çökmesi, mm). "hastalık", anjiyogramda en az bir ana koroner arterde çapın yüzde 50'sinden fazla daralma görüldüyse 1, aksi hâlde 0 olur.  
> Görev: Referansın girdilerini hazırla. Girdiler yaş, erkekler için 1 ve kadınlar için 0 olan bir sütun ve atipik anjina, anjina dışı ağrı ve ağrı yok için 0 ile 1 değerlerinden oluşan üç sütundur; böylece tipik anjina referans kategori olur. Her girdiyi eğitim hastalarının ortalaması ve standart sapmasıyla bir z-skoruna dönüştür ve hastalık için olağan lojistik regresyonu, standart ayarlarıyla ve en fazla 1.000 yinelemeyle, yalnızca eğitim hastalarında kur. Tam model aynı girdileri kullanır ve bunlara istirahat kan basıncını, kolesterolü, en yüksek kalp hızını, egzersiz anjinasını ve ST çökmesini ekler; aynı yolla kurulur. Her test hastası için iki modelin hastalık olasılığını ve test hastalarındaki AUC değerlerini hesapla. Test hastalarından 2.000 yeniden örneklem çek: Her yeniden örneklem, 2026 seed değeriyle başlatılan bir rastgele sayı üreteciyle, tablodaki hasta sayısı kadar hastayı yerine koyarak çeker. Yalnızca tek bir sonucun hastalarını içeren yeniden örneklemi atla. Her yeniden örneklemde aynı hastalarda referansın ve modelin ROC eğrisi altındaki alanını (AUC) ve farklarını (model eksi referans) hesapla. Bir sonucun yüzde 95'lik aralığı, değerlerinin yeniden örneklemler boyunca 2,5. yüzdeliğinden 97,5. yüzdeliğine uzanır.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: İki modelin ve farkın AUC değerlerini aralıklarıyla birlikte bir tabloda göster; farkın 2,5. yüzdeliğini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Hiçbir test hastası iki modelden birini kurmak için kullanılmamalı ve iki model aynı yeniden örneklemlerde değerlendirilmeli.  

In [ ]:
#@title Proje adımı 2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _patients("both")
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 3: Kalibrasyon ve karar eşiği

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kardiyolog, tam modelin dışlama eşiğini eğitim hastalarında seçiyor ve test hastaları açılmadan önce kalibrasyonu denetliyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda UCI Heart Disease veri setinin Cleveland verisinden, yani bir egzersiz testinden sonra koroner anjiyografiye gönderilmiş hastalardan 303 hasta bulunur. Her satır bir hastadır: "hasta" hastanın verideki numarasını (0-302), "küme" 2. haftanın kümesini verir ("eğitim", 227 hasta; "test", 76 hasta). Klinik değişkenler şunlardır: "yaş" (yıl), "cinsiyet" ("erkek" ya da "kadın"), "göğüs ağrısı tipi" ("tipik anjina", "atipik anjina", "anjina dışı ağrı" ya da "ağrı yok"), "istirahat kan basıncı" (mmHg), "kolesterol" (mg/dL), "en yüksek kalp hızı" (egzersiz testinde ulaşılan en yüksek kalp hızı, dakikada), "egzersiz anjinası" (egzersiz testi anjinaya yol açtıysa 1) ve "ST çökmesi" (egzersizde ST segmentinin istirahate göre çökmesi, mm). "hastalık", anjiyogramda en az bir ana koroner arterde çapın yüzde 50'sinden fazla daralma görüldüyse 1, aksi hâlde 0 olur.  
> Görev: 2. adımdaki tam modeli eğitim hastalarında beş katlı çapraz doğrulamayla kur; katlar aynı hastalık oranıyla ve 2026 seed değeriyle çekilsin; böylece her eğitim hastası onu görmemiş bir modelin olasılığını alsın. Bu olasılıklarda kalibrasyon sabitini ve eğimini 2. duraktaki gibi hesapla; 0,01, 0,02 ve bu biçimde 0,50'ye kadar giden eşikler arasından, hastalığı olan eğitim hastalarının en az yüzde 95'inin olasılığının ona eşit ya da ondan büyük olduğu en yüksek eşiği seç. Ardından tam modeli bütün eğitim hastalarında kur, eşiği test hastalarına bir kez uygula ve dışlanan hastaları hastalığı olan ve olmayan olarak say.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Çapraz doğrulama olasılıklarının kalibrasyonunu, eşiği ve test hastalarındaki sayıları yazdır; dışlanan test hastalarının sayısını "cevap" adıyla sakla.  
> Test: Eşik herhangi bir test hastası kullanılmadan seçilmeli ve eşikteki eğitim duyarlılığı en az 0,95 olmalı.  

In [ ]:
#@title Proje adımı 3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _patients("both")
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 4: Rapor için sonuç tablosu

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Final projesinin raporu TRIPOD+AI'ı izler: Her sonuç, hem bütün hastalarda hem de modelin kullanılacağı gruplarda aralığıyla birlikte verilir.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda UCI Heart Disease veri setinin Cleveland verisinden, yani bir egzersiz testinden sonra koroner anjiyografiye gönderilmiş hastalardan 76 test hastası bulunur. Her satır bir hastadır: "hasta" hastanın verideki numarasını, "cinsiyet" "erkek" ya da "kadın" değerini, "olasılık, referans" dersin bir referans modelinin (yaş, cinsiyet ve göğüs ağrısı tipiyle kurulan bir lojistik regresyon) verdiği koroner arter hastalığı olasılığını, "olasılık, model" ise dersin tam modelinin (buna istirahat kan basıncı, kolesterol ve egzersiz testinin üç sonucu eklenmiş hâli) verdiği olasılığı gösterir. İki model de yalnızca 227 eğitim hastasında kurulmuştur. "hastalık", anjiyogramda en az bir ana koroner arterde çapın yüzde 50'sinden fazla daralma görüldüyse 1, aksi hâlde 0 olur. Projenin eğitim hastalarında seçilen dışlama eşiği 0,18 değerindedir.  
> Görev: Test hastalarından 2.000 yeniden örneklem çek: Her yeniden örneklem, 2026 seed değeriyle başlatılan bir rastgele sayı üreteciyle, tablodaki hasta sayısı kadar hastayı yerine koyarak çeker. Yalnızca tek bir sonucun hastalarını içeren yeniden örneklemi atla. Her yeniden örneklemde aynı hastalarda referansın ve modelin ROC eğrisi altındaki alanını (AUC) ve farklarını (model eksi referans) hesapla. Bir sonucun yüzde 95'lik aralığı, değerlerinin yeniden örneklemler boyunca 2,5. yüzdeliğinden 97,5. yüzdeliğine uzanır. Ardından 0,18 dışlama eşiğinde duyarlılığı, özgüllüğü ve negatif öngörü değerini bütün test hastaları, kadınlar ve erkekler için hesapla. Her oranı yüzde 95'lik Wilson aralığıyla ver: n hastanın k tanesi için, z = 1,96 olmak üzere merkez (k/n artı z kare bölü 2n) bölü (1 artı z kare bölü n), yarı genişlik ise z çarpı (k/n çarpı bir eksi k/n, bölü n, artı z kare bölü 4n kare) değerinin karekökü, bölü (1 artı z kare bölü n) olur. Bütün sonuçları sonuç, grup, değer, alt sınır ve üst sınır sütunlarından oluşan ve rapora kopyalanmaya hazır tek bir tabloya yaz.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Tabloyu göster; bütün test hastalarındaki negatif öngörü değerinin alt sınırını üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Her aralık kendi değerini içermeli ve 0 ile 1 arasında kalmalı; yalnızca AUC farkının aralığı negatif olabilir.  

In [ ]:
#@title Proje adımı 4 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_c = _course()
veri = pd.DataFrame({"hasta": _c["X_test"].index, "cinsiyet": _c["X_test"]["sex"].map({1: "erkek", 0: "kadın"}).to_numpy(),
                     "hastalık": _c["y_test"].to_numpy(), "olasılık, referans": np.round(_c["p_ref"], 6),
                     "olasılık, model": np.round(_c["p_model"], 6)})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

<a id="part-2"></a>

# Kısım 2: Python yolu

**Kimler için?** Bu kısım Python yazmayı öğrenmek isteyen katılımcılar içindir. Kısım 1'i çalışmış olmanız gerekmez: 0. bölümden sonra doğrudan buradan başlayın.

**Nasıl ilerler?** Hücreleri yukarıdan aşağıya Shift+Enter ile çalıştırın. Baştaki *Python adımı*, haftanın gerektirdiği dil bilgisini öğretir. Numaralı bölümlerdeki alıştırmalar hemen *doğru* ya da *henüz değil* yazar; örnek çözümler katlanmış hücrelerde durur ve ancak denedikten sonra açılmalıdır. *Gözlemle ve düşün* hücreleri açık uçlu sorular sorar; bu soruların çözüm hücresi yoktur. *Python notu* hücreleri, yeni bir yapının ilk kullanıldığı yerde dili açıklar. *İleri düzey (isteğe bağlı)* olarak işaretlenen hücreler, programlama bilenler için ek çalışmalardır. Bu kısma ortadan başlayan katılımcı, o bölümün başındaki *Telafi* hücresini çalıştırır. Bazı alıştırma hücreleri `veri` adlı bir tabloyu hazırlayan bir satırla başlar. Bu tablo, etkileşimli labın prompt atölyesindeki alıştırma promptları için hazırlanmıştır; Python yolunda alıştırma, metninde adı geçen tablolarla çözülebilir.

## Python adımı 11: Katılımcının kendi projesi

Bu hafta dersin kodu sizin kendi projenizin koduna dönüşür. Üç alışkanlık başkalarının denetleyebileceği bir proje ortaya çıkarır: Her dosyanın bir yeri olan bir klasör, ne yaptığını söyleyen fonksiyonlar ve defter her çalıştığında aynı çıkan sonuçlar. Bu adımda `pathlib` modülünün bir projenin klasörlerini nasıl oluşturduğu, bir *docstring* ile bir fonksiyonun nasıl belgelendiği, bir *seed değeriyle* başlatılan rastgele sayı üretecinin tekrarlanabilen bir yeniden örneklemi nasıl çektiği ve paketlerin sürümlerinin sonuçların yanına nasıl yazıldığı gösterilmektedir.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
project = pathlib.Path("final_project")
for name in ["data", "results"]:
    (project / name).mkdir(parents=True, exist_ok=True)
readme = project / "README.md"
if not readme.exists():
    readme.write_text("# Egzersiz testinden sonra koroner arter hastalığını dışlamak\n", encoding="utf-8")
print(sorted(entry.name for entry in project.iterdir()))

**Python notu.** `pathlib.Path("final_project")` bir yoldur, henüz bir klasör değildir. `/` işleci yolları birleştirir; böylece `project / "data"` data alt klasörünü gösterir. `mkdir(parents=True, exist_ok=True)` klasörü ve üstündeki klasörleri oluşturur; klasör zaten varsa hiçbir şey yapmaz, bu yüzden hücre iki kez çalışabilir. `readme.exists()` dosyanın orada olup olmadığını `write_text` onu yazmadan önce sorar; böylece düzenlediğiniz bir README dosyasının üzerine hiçbir zaman yazılmaz. `project.iterdir()` klasördeki öğeleri listeler, `sorted(...)` da adlarını sıraya koyar.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
def share_with_disease(outcomes):
    """Sonucu 1 olan hastaların payı.

    outcomes: Her hasta için bir değer olmak üzere 0 ve 1'lerden oluşan bir liste.
    0 ile 1 arasında bir sayı döndürür."""
    return sum(outcomes) / len(outcomes)


p10 = [0.047, 0.092, 0.168, 0.198, 0.304, 0.466, 0.608, 0.691, 0.936, 0.978]
y10 = [0, 0, 0, 1, 0, 1, 0, 1, 1, 1]
print(share_with_disease(y10))
help(share_with_disease)

**Python notu.** `def` satırının hemen altındaki üç tırnaklı metin fonksiyonun *docstring* metnidir: Fonksiyonun ne yaptığını, ne beklediğini ve ne döndürdüğünü söyler. `help(share_with_disease)`, NumPy ya da pandas fonksiyonlarının yardımını yazdırdığı gibi bu metni de yazdırır. Docstring belgelemenin en kısa biçimidir ve kodla birlikte taşınır. `p10` ve `y10`, ders sayfasındaki bootstrap çözümlü örneğinin on hastasının olasılıkları ve sonuçlarıdır.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
rng = np.random.default_rng(2026)
draw = rng.integers(0, 10, 10)
again = np.random.default_rng(2026).integers(0, 10, 10)
print(draw, again, bool((draw == again).all()))
print("yeniden örneklemin hastaları, 1'den numaralanmış:", sorted(int(k) + 1 for k in draw))
print("aynı üretecin ikinci çekilişi:", rng.integers(0, 10, 10))

**Python notu.** `np.random.default_rng(2026)` 2026 seed değeriyle başlatılan bir rastgele sayı üreteci oluşturur. `rng.integers(0, 10, 10)` 0 ile 9 arasında on tam sayı çeker; üst sınır olan 10 dâhil değildir. Aynı seed değeriyle başlatılan ikinci bir üreteç tam olarak aynı sayıları çeker; bu yüzden seed değeri sabitlenmiş bir bootstrap her bilgisayarda aynı aralığı verir. Üreteç her çekilişte ilerler: İkinci çekilişi birincisinden farklıdır ve 2.000 yeniden örneklemli bir bootstrap yönteminin ihtiyaç duyduğu da budur. Burada çekilen sayılar dersteki çözümlü örneğin yeniden örneklemidir: P1, P1, P2, P4, P4, P4, P5, P7, P7 ve P9.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
summary = {"hasta": len(y10), "hastalık oranı": share_with_disease(y10), "seed": 2026,
           "python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__}
(project / "results" / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(summary, indent=2, ensure_ascii=False))

**Python notu.** Bir sonuç ancak hesaplandığı koşullarla birlikte yeniden üretilebilir. `platform.python_version()` Python'un sürümünü, `np.__version__`, `pd.__version__` ve `sklearn.__version__` ise paketlerin sürümlerini verir. `summary` sözlüğü bunları sonuç ve seed değeriyle birlikte tutar; `json.dumps(..., indent=2)` sözlüğü her programın okuyabileceği bir metin olarak `final_project/results/summary.json` dosyasına yazar. `ensure_ascii=False` Türkçe harflerin olduğu gibi kalmasını sağlar.

**Hızlı kontrol.** Sonuç dosyası neden rastgele sayıların seed değerini ve paketlerin sürümlerini tutar?

<details><summary>Cevap</summary>

Başka bir kişinin ya da bir yıl sonra sizin aynı kodla aynı yeniden örneklemleri çekip aynı aralıkları elde edebilmesi için. Bir sonuç yine de farklı çıkarsa, sürümler arada bir paketin değişip değişmediğini gösterir.

</details>

**Alıştırma 11.1.** Yukarıdaki gibi 2026 seed değeriyle başlatılan bir üreteçle 0 ile 9 arasında on tam sayı çekin. Yeniden örneklemde kaç farklı hasta vardır?

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma 11.1", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = len(set(np.random.default_rng(2026).integers(0, 10, 10).tolist()))
teach.check("Alıştırma 11.1", cevap)

**Alıştırma 11.2.** `project` klasörüne data ve results klasörlerinin yanına code ve figures alt klasörlerini ekleyin. final_project klasöründe kaç öğe vardır?

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma 11.2", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
for name in ["data", "code", "results", "figures"]:
    (project / name).mkdir(parents=True, exist_ok=True)
cevap = len(list(project.iterdir()))
teach.check("Alıştırma 11.2", cevap)

**Alıştırma 11.3.** Hastalığı olan ve olmayan birer hastadan oluşan çiftlerde hastalığı olan hastanın olasılığının daha yüksek olduğu çiftlerin payını, eşitlikleri yarım sayarak döndüren ve bir docstring içeren `auc_pairs(y, p)` fonksiyonunu yazın. `p10` ve `y10` ile verilen on hastanın `draw` yeniden örnekleminin AUC değeri nedir? Üç ondalık basamakla saklayın.

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma 11.3", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
def auc_pairs(y, p):
    """Doğru sıradaki (hastalık var, hastalık yok) çiftlerinin payı; eşitlik yarım sayılır."""
    pos = [pi for yi, pi in zip(y, p) if yi == 1]
    neg = [pi for yi, pi in zip(y, p) if yi == 0]
    return sum(1.0 if a > b else 0.5 if a == b else 0.0 for a in pos for b in neg) / (len(pos) * len(neg))


cevap = round(auc_pairs([y10[k] for k in draw], [p10[k] for k in draw]), 3)
teach.check("Alıştırma 11.3", cevap)

## 1. Soru ve verilerin denetimi

*Ders: [Veriler: Kaynak, izin ve denetim](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-11/ders.html#the-data-source-permission-and-audit)*

Çözümlü final projesi, egzersiz testinden sonra bilinen rutin değişkenlerin koroner arter hastalığını dışlayıp anjiyografiye sevki önleyip önleyemeyeceğini sorar. Veriler, eğitimin 1. haftadan beri kullandığı 303 Cleveland hastasına aittir. Denetim her sütunun eksik değerlerini sayar ve projenin kullandığı sütunları, yani karar anında bilinen sekiz değişkeni ve sonucu işaretler.

**Dersten koda.** Projenin sekiz değişkeni: `FEATURES`. Her sütunun eksik değerleri: `missing`. Denetimin tablosu: `audit`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 1. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB11_final_projesi_atolyesi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-11/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(1)

In [ ]:
heart = load_cleveland()
FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
missing = heart.isna().sum()
audit = pd.DataFrame({"eksik değer": missing, "projede kullanılıyor": [c in FEATURES for c in missing.index]})

print(audit)
print(len(heart), "hasta,", int(heart["disease"].sum()), "hastalıklı")

**Python notu.** `heart.isna()` aynı biçimde ve değerin eksik olduğu yerlerde `True` içeren bir tablo verir; `.sum()` her sütundaki `True` değerlerini sayar. `[c in FEATURES for c in missing.index]` *liste üreteci* (list comprehension) her sütun için o sütunun projeye ait olup olmadığını sorar. `pd.DataFrame({...})` iki sütunu, verinin sütun adlarını indeks olarak kullanarak yan yana koyar.

In [ ]:
share = heart.groupby("cp")["disease"].agg(["count", "mean"])
share.index = ["tipik anjina", "atipik anjina", "anjina dışı ağrı", "ağrı yok"]
print(share.round(3))
fig, ax = plt.subplots(figsize=(7.2, 2.8))
ax.barh(share.index, share["mean"], color="#1F5F8B")
ax.set_xlim(0, 1)
ax.set_xlabel("anjiyogramda hastalık oranı")
ax.set_title("Göğüs ağrısı tipine göre hastalık, 303 Cleveland hastası (UCI Heart Disease, CC BY 4.0)", fontsize=9)
plt.tight_layout()
plt.show()

**Gözlemle ve düşün.** Bu verilerde göğüs ağrısı olmayan hastalar en yüksek, tipik anjinası olan hastalar ise en düşük hastalık oranlarından birine sahiptir; bu, bir göğüs ağrısı polikliniğinin test öncesi olasılığının beklediğinin tersidir. Bu durum hastaların anjiyografiye nasıl geldikleri hakkında ne söyler ve bunu öğrenen bir model sizin kliniğinizde ne yapardı?

**Alıştırma A.** 303 hastada hastalık oranı nedir? Oranı üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("A")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma A", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float(heart["disease"].mean()), 3)
teach.check("Alıştırma A", cevap)

## 2. Hasta düzeyinde ayırma

*Ders: [Veriler: Kaynak, izin ve denetim](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-11/ders.html#the-data-source-permission-and-audit)*

Proje 2. haftanın ayırmasını korur: Hastaların dörtte biri, iki kümede aynı hastalık oranıyla test için ayrılır. Cleveland verisinin her satırı bir hasta olduğu için satırların ayrılması hastaların ayrılması demektir; hasta başına birkaç satır olsaydı bir hastanın bütün satırlarının aynı kümede kalması gerekirdi. Göğüs ağrısı tipi, tipik anjina referans kategori olmak üzere 0 ve 1 değerlerinden oluşan üç sütuna dönüşür.

**Dersten koda.** Girdiler ve sonuç: `X`, `y`. Eğitim ve test hastaları: `X_train`, `X_test`, `y_train`, `y_test`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 2. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB11_final_projesi_atolyesi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-11/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(2)

In [ ]:
cad = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"], drop_first=True, dtype=int)
X, y = cad.drop(columns="disease"), cad["disease"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

print(pd.DataFrame({"hasta": [len(y_train), len(y_test)], "hastalıklı": [int(y_train.sum()), int(y_test.sum())]},
                   index=["eğitim", "test"]))
print(list(X.columns))

**Python notu.** `pd.get_dummies(..., columns=["cp"], drop_first=True, dtype=int)` 1 ile 4 arasında kodlanmış göğüs ağrısı tipini `cp_2.0`, `cp_3.0` ve `cp_4.0` sütunlarına dönüştürür; `drop_first=True` ilk kategoriyi, yani referans olacak tipik anjinayı dışarıda bırakır. `train_test_split(..., stratify=y, random_state=RANDOM_STATE)` test için ayrılan dörtte biri, iki kümede de verinin bütünündeki hastalık oranıyla ve dersin 2026 seed değeriyle çeker; böylece ayırma her hafta ve her bilgisayarda aynı olur.

**Gözlemle ve düşün.** Test kümesinde 35'i hastalıklı olmak üzere 76 hasta vardır. Ders sayfasının bootstrap bölümü, bir duyarlılığın alt sınırının ancak hastalığı olan en az 73 hastanın hepsi bulunduğunda 0,95 olabileceğini gösterir. Bu, projenin test kümesinin neyi gösterebileceği için ne anlama gelir?

**Alıştırma B.** Test hastalarının kaçında hastalık vardır?

In [ ]:
veri = alistirma_tablosu("B")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma B", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = int(y_test.sum())
teach.check("Alıştırma B", cevap)

## 3. Ne kadar veri

*Ders: [Ne kadar veri: Riley ve arkadaşlarının ölçütleri](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-11/ders.html#how-much-data-the-criteria-of-riley-and-colleagues)*

Riley ve arkadaşlarının üç ölçütü, iki sonuçlu bir model için en küçük hasta sayısını verir. Ölçütler parametre sayısına, sonucun oranına ve öngörülen bir Cox-Snell R kare değerine ihtiyaç duyar; proje bu değeri yaş, cinsiyet ve göğüs ağrısı tipinden oluşan referansın eğitim hastalarındaki değerinden alır.

**Formüllerden koda.** $R^2_{CS}$: `cox_snell_r2`. $n_{(i)}$, $n_{(ii)}$, $n_{(iii)}$ ve $R^2_{\max}$: `riley_sizes`. $P$: `P`. Öngörülen $R^2_{CS}$: `r2_cs`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 3. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB11_final_projesi_atolyesi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-11/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(3)

In [ ]:
def cox_snell_r2(y, p):
    """Tahmin edilen p olasılıklarının y sonuçları için Cox-Snell R kare değerini hesaplar."""
    y, p = np.asarray(y, dtype=float), np.clip(np.asarray(p, dtype=float), 1e-12, 1 - 1e-12)
    phi = y.mean()
    ll_null = len(y) * (phi * np.log(phi) + (1 - phi) * np.log(1 - phi))
    ll_model = np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
    return 1 - np.exp(2 * (ll_null - ll_model) / len(y))


def riley_sizes(P, prevalence, r2_cs, shrinkage=0.9, delta=0.05, margin=0.05):
    """Riley ve arkadaşlarının P parametreli bir ikili sonuç modeli için önerdiği üç büyüklüğü hesaplar. Ölçütler (i) en az 0,9 büzülme, (ii) görünen ve düzeltilmiş Nagelkerke R kare değerleri arasında en fazla 0,05 fark ve (iii) genel riskin 0,05 içinde kestirilmesidir."""
    r2_max = 1 - np.exp(2 * (prevalence * np.log(prevalence) + (1 - prevalence) * np.log(1 - prevalence)))
    n1 = P / ((shrinkage - 1) * np.log(1 - r2_cs / shrinkage))
    s2 = r2_cs / (r2_cs + delta * r2_max)
    n2 = P / ((s2 - 1) * np.log(1 - r2_cs / s2))
    n3 = (1.96 / margin) ** 2 * prevalence * (1 - prevalence)
    return {"(i)": math.ceil(n1), "(ii)": math.ceil(n2), "(iii)": math.ceil(n3), "R2_max": round(float(r2_max), 3)}

REFERENCE = ["age", "sex", "cp_2.0", "cp_3.0", "cp_4.0"]
anticipated = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[REFERENCE], y_train)
r2_cs = cox_snell_r2(y_train, anticipated.predict_proba(X_train[REFERENCE])[:, 1])
P = X_train.shape[1]
sizes = riley_sizes(P, float(y_train.mean()), r2_cs)

print("P =", P, "| oran", round(float(y_train.mean()), 3), "| R² (Cox-Snell)", round(float(r2_cs), 3))
print(sizes, "| eğitim hastası", len(y_train))

**Python notu.** `cox_snell_r2` iki log-olabilirliği karşılaştırır; bunlardan biri `p` olasılıklarına, öteki her hastaya genel `phi` oranını veren bir modele aittir. `np.clip` her olasılığı logaritmanın sonsuz olacağı tam 0 ve 1 değerlerinden uzak tutar. `riley_sizes` dersteki üç formülü satır satır yazar; `math.ceil` de her büyüklüğü tam bir hastaya yukarı yuvarlar. `shrinkage`, `delta` ve `margin` argümanlarının *varsayılan değerleri* vardır; bu yüzden `riley_sizes(P, prevalence, r2_cs)` başka değerler verilmedikçe 0,9, 0,05 ve 0,05 kullanır.

In [ ]:
grid = np.linspace(0.05, 0.6, 56)
fig, ax = plt.subplots(figsize=(7.2, 3.2))
for key, colour in (("(i)", "#9B1B1B"), ("(ii)", "#B97813"), ("(iii)", "#0E7A78")):
    ax.plot(grid, [riley_sizes(P, float(y_train.mean()), r)[key] for r in grid], color=colour, label=key)
ax.axhline(len(y_train), color="#231A1A", linestyle="--", linewidth=1, label="eğitim hastası")
ax.axvline(r2_cs, color="#8A8A8A", linestyle=":", linewidth=1)
ax.set_ylim(0, 800)
ax.set_xlabel("öngörülen Cox-Snell R kare")
ax.set_ylabel("gereken hasta")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Alıştırma C.** Model yalnızca referansın beş parametresine sahip olsaydı, aynı R kare ve hastalık oranıyla (i) ölçütü kaç hasta isterdi?

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma C", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = riley_sizes(5, float(y_train.mean()), r2_cs)["(i)"]
teach.check("Alıştırma C", cevap)

## 4. Bir referans ve model

*Ders: [Önce bir referans](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-11/ders.html#a-reference-first)*

Referans, kardiyoloğun egzersiz testinden önce bildikleriyle, yani yaş, cinsiyet ve göğüs ağrısı tipiyle kurulan bir lojistik regresyondur. Tam model buna istirahat kan basıncını, kolesterolü ve testin üç sonucunu ekler. İkisi de aynı işlem hattıyla eğitim hastalarında kurulur ve aynı test hastalarında bir kez sınanır.

**Dersten koda.** Referansın ve modelin sütunları: `REFERENCE`, `ALL`. Kurulmuş iki model: `reference`, `model`. Test hastaları için olasılıkları: `p_ref`, `p_model`. Test hastalarının sonuçları bir dizi olarak: `yt`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 4. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB11_final_projesi_atolyesi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-11/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(4)

In [ ]:
ALL = list(X.columns)


def fit(columns):
    """Eğitim hastalarında bir lojistik regresyon kurar; ölçekleme işlem hattının içinde yer alır ve yalnızca eğitim hastalarından öğrenilir."""
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[columns], y_train)


reference, model = fit(REFERENCE), fit(ALL)
p_ref = reference.predict_proba(X_test[REFERENCE])[:, 1]
p_model = model.predict_proba(X_test[ALL])[:, 1]
yt = y_test.to_numpy()

print("test AUC, referans", round(roc_auc_score(yt, p_ref), 3), "| test AUC, model", round(roc_auc_score(yt, p_model), 3))
fig, ax = plt.subplots(figsize=(4.6, 4.2))
for name, p, colour in (("referans", p_ref, "#8A8A8A"), ("model", p_model, "#9B1B1B")):
    fpr, tpr, _ = roc_curve(yt, p)
    ax.plot(fpr, tpr, color=colour, label=f"{name}, AUC {roc_auc_score(yt, p):.3f}")
ax.plot([0, 1], [0, 1], color="#C8C0B8", linewidth=1)
ax.set_xlabel("1 − özgüllük")
ax.set_ylabel("duyarlılık")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Python notu.** `fit(columns)` kurulmuş bir işlem hattı döndüren küçük bir fonksiyondur; böylece referans ve model, iki sütun listesiyle aynı satır tarafından kurulur. `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))` kurulurken sütunların ortalamalarını ve standart sapmalarını eğitim hastalarından öğrenir, tahmin ederken de bunları test hastalarına uygular. `predict_proba(...)[:, 1]` olasılıkların ikinci sütununu, yani hastalığın olasılığını alır.

**Gözlemle ve düşün.** İki ROC eğrisi birkaç yerde kesişir ve birbirine değer. Yalnızca eğrilere bakarak tam modelin yeni hastalarda referanstan daha iyi olup olmadığını söyleyebilir miydiniz?

**Alıştırma D.** Referans sütunları ve egzersiz testinin üç sonucuyla, kan basıncı ve kolesterol olmadan kurulan bir model testte hangi AUC değerine ulaşır? Üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("D")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma D", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
exercise_only = fit(REFERENCE + ["thalach", "exang", "oldpeak"])
cevap = round(float(roc_auc_score(yt, exercise_only.predict_proba(X_test[REFERENCE + ["thalach", "exang", "oldpeak"]])[:, 1])), 3)
teach.check("Alıştırma D", cevap)

## 5. Belirsizlik: Bootstrap

*Ders: [Belirsizlik: Bootstrap](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-11/ders.html#uncertainty-the-bootstrap)*

Bootstrap yöntemi test hastalarını yerine koyarak yeniden çeker, iki modelin AUC değerlerini aynı yeniden örneklemde hesaplar ve bunu 2.000 kez tekrarlar. Sonuçların 2,5. ve 97,5. yüzdelikleri yüzde 95'lik aralıkları verir; iki AUC değerinin farkı her yeniden örneklemde hesaplandığı için farkın aralığı projenin sorusunu cevaplar. Wilson aralığı bir oranın belirsizliğini verir.

**Formüllerden koda.** Bir yeniden örneklem: `i`. Her yeniden örneklemde AUC değerleri ve farkları: `boot`. $[q_{0{,}025}, q_{0{,}975}]$ yüzdelik aralığı: `interval`. Wilson aralığı: `wilson`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 5. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB11_final_projesi_atolyesi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-11/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(5)

In [ ]:
def bootstrap_auc(y, p_a, p_b, n_boot=2000, seed=RANDOM_STATE):
    """İki modelin AUC değerlerini ve farklarını, test hastalarından yerine koyarak çekilen n_boot yeniden örneklemde hesaplar."""
    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    rows = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() == y[i].max():
            continue
        a, b = roc_auc_score(y[i], p_a[i]), roc_auc_score(y[i], p_b[i])
        rows.append({"a": a, "b": b, "difference": b - a})
    return pd.DataFrame(rows)


def interval(values, level=0.95):
    """Bootstrap değerlerinin yüzdelik aralığını verir."""
    return tuple(np.percentile(values, [50 * (1 - level), 100 - 50 * (1 - level)]))


def wilson(k, n, z=1.96):
    """Bir k/n oranının Wilson aralığını verir."""
    p = k / n
    centre = (p + z ** 2 / (2 * n)) / (1 + z ** 2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / (1 + z ** 2 / n)
    return centre - half, centre + half

boot = bootstrap_auc(yt, p_ref, p_model)
print(len(boot), "yeniden örneklem")
for column in ["a", "b", "difference"]:
    low_, high_ = interval(boot[column])
    print(f"{column:>10}: {boot[column].mean():.3f} [{low_:.3f}, {high_:.3f}]")
print("modelin daha iyi olmadığı pay", round(float((boot["difference"] <= 0).mean()), 3))
print("duyarlılık 35'te 35, Wilson", np.round(wilson(35, 35), 3))

**Python notu.** `bootstrap_auc` içinde `rng.integers(0, len(y), len(y))` hasta sayısı kadar konumu yerine koyarak çeker; `y[i]`, `p_a[i]` ve `p_b[i]` üç diziden de aynı hastaları seçer: İki model her zaman aynı yeniden örneklemde değerlendirilir. Bütün hastaların aynı sonuca sahip olduğu bir yeniden örneklemin AUC değeri olmaz; `if y[i].min() == y[i].max(): continue` böyle bir yeniden örneklemi atlar. `rows.append({...})` her yeniden örneklem için bir sözlük toplar, `pd.DataFrame(rows)` de `a`, `b` ve `difference` sütunlarından oluşan `boot` tablosunu kurar. `np.percentile(values, [2.5, 97.5])` aralığın iki ucunu verir.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.2))
axes[0].hist(boot["a"], bins=40, color="#8A8A8A", alpha=0.7, label="referans")
axes[0].hist(boot["b"], bins=40, color="#9B1B1B", alpha=0.6, label="model")
axes[0].set_xlabel("AUC")
axes[0].legend(fontsize=8)
axes[1].hist(boot["difference"], bins=40, color="#1F5F8B")
for v in interval(boot["difference"]):
    axes[1].axvline(v, color="#231A1A", linestyle="--", linewidth=1)
axes[1].axvline(0, color="#231A1A", linewidth=1.2)
axes[1].set_xlabel("modelin AUC değeri eksi referansın AUC değeri")
plt.tight_layout()
plt.show()

**Gözlemle ve düşün.** Farkın histogramının sıfırın solunda kalan bir bölümü vardır. Final projesinin egzersiz testinin sonuçları hakkında neyi gösterdiğini ve neyi göstermediğini raporun tek bir cümlesinde nasıl söylerdiniz?

**İleri düzey (isteğe bağlı).** `bootstrap_auc(yt, p_ref, p_model, seed=1)` komutunu çalıştırın ve aralıklarını 2026 seed değeriyle bulunanlarla karşılaştırın. Aralıklar üçüncü ondalık basamakta değişir: Hangi yeniden örneklemlerin çekileceğine seed değeri karar verir, ancak aralığın genişliği 76 hastadan gelir.

**Alıştırma E.** Yeniden örneklemlerin hangi payında tam model referanstan daha iyi değildir? Payı üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("E")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma E", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float((boot["difference"] <= 0).mean()), 3)
teach.check("Alıştırma E", cevap)

## 6. Kalibrasyon

*Ders: [Kalibrasyon: Olasılıklar tutuyor mu?](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-11/ders.html#calibration-do-the-probabilities-hold)*

Sonuçların modelin log-odds değerleri üzerine kurulan iki lojistik regresyonu kalibrasyonu ölçer. Log-odds değerleri sabit bir kayma olarak alındığında birinci regresyon kalibrasyon sabitini, ikinci regresyon ise kalibrasyon eğimini verir. Birkaç Newton adımı bunları hiçbir ceza terimi olmadan kurar. Bootstrap yöntemi 76 test hastasında eğimin aralığını verir.

**Formüllerden koda.** $\mathrm{LP}$: `lp`. $a$ sabiti ve $b$ eğimi: `calibration` fonksiyonunun döndürdüğü iki değer. Eğimin aralığı: `slope_interval`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 6. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB11_final_projesi_atolyesi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-11/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(6)

In [ ]:
def calibration(y, p, steps=50):
    """Tahmin edilen p olasılıklarının kalibrasyon sabitini (doğrusal öngörücü offset olarak alınarak) ve eğimini Newton adımlarıyla hesaplar."""
    y = np.asarray(y, dtype=float)
    lp = np.log(p / (1 - p))
    a = 0.0
    for _ in range(steps):
        q = 1 / (1 + np.exp(-(a + lp)))
        a += (y - q).sum() / (q * (1 - q)).sum()
    Z = np.c_[np.ones_like(lp), lp]
    b = np.zeros(2)
    for _ in range(steps):
        q = 1 / (1 + np.exp(-Z @ b))
        b += np.linalg.solve(Z.T @ (Z * (q * (1 - q))[:, None]), Z.T @ (y - q))
    return float(a), float(b[1])


def slope_interval(y, p, n_boot=1000, seed=RANDOM_STATE):
    """Kalibrasyon eğiminin bootstrap yüzdelik aralığını verir."""
    rng = np.random.default_rng(seed)
    y, p = np.asarray(y), np.asarray(p)
    slopes = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() < y[i].max():
            slopes.append(calibration(y[i], p[i])[1])
    return interval(slopes)

a_test, b_test = calibration(yt, p_model)
print("test: sabit", round(a_test, 3), "| eğim", round(b_test, 3), "| eğimin aralığı", np.round(slope_interval(yt, p_model), 3))
print("ortalama olasılık", round(float(p_model.mean()), 3), "| gözlenen oran", round(float(yt.mean()), 3))
groups = pd.qcut(p_model, 5, labels=False)
fifths = pd.DataFrame({"tahmin": pd.Series(p_model).groupby(groups).mean(), "gözlenen": pd.Series(yt).groupby(groups).mean(),
                       "hasta": pd.Series(yt).groupby(groups).size()}).round(3)
print(fifths)
fig, ax = plt.subplots(figsize=(4.4, 4.2))
ax.plot([0, 1], [0, 1], color="#C8C0B8", linewidth=1)
ax.plot(fifths.iloc[:, 0], fifths.iloc[:, 1], "o-", color="#9B1B1B")
ax.set_xlabel("ortalama tahmin edilen olasılık")
ax.set_ylabel("gözlenen hastalık oranı")
plt.tight_layout()
plt.show()

**Python notu.** Önce sabit kurulur: `q`, `a` kaymasıyla elde edilen olasılıklardır ve her adım `y - q` artıklarının toplamının `q * (1 - q)` ağırlıklarının toplamına bölümünü ekler. Eğim için `Z` birlerden oluşan bir sütun ile log-odds değerlerini tutar; her Newton adımı `np.linalg.solve` ile iki denklemli küçük bir sistemi çözer. Elli adım gerekenden çok fazladır; sonuç birkaç adımdan sonra değişmez. `pd.qcut(p_model, 5, labels=False)` hastaları olasılıklarına göre aynı büyüklükte beş gruba ayırır.

**Gözlemle ve düşün.** Test hastalarında eğimin aralığı 1'in altından 2'nin üstüne uzanır. Modelin kalibrasyonu olarak rapora hangi sayıyı, hangi kelimelerle koyardınız?

**Alıştırma F.** Referansın test hastalarındaki kalibrasyon eğimi nedir? Üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("F")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma F", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(calibration(yt, p_ref)[1], 3)
teach.check("Alıştırma F", cevap)

## 7. Bir dışlama eşiği

*Ders: [Bir karar eşiği: Dışlama](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-11/ders.html#a-decision-threshold-ruling-out)*

Beş katlı çapraz doğrulama her eğitim hastasına o hastayı görmemiş bir modelin olasılığını verir. Proje 0,01 ile 0,50 arasındaki bir ızgarada, hastalığı olan eğitim hastalarının en az yüzde 95'inin eşikte ya da eşiğin üstünde kaldığı en yüksek eşiği alır. Eşik ardından test hastalarına bir kez uygulanır.

**Formüllerden koda.** Çapraz doğrulama olasılıkları: `p_cv`. Eşik: `threshold`. Dışlanan hastalar: `ruled_out`. TP, FN, TN ve FP: `counts`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 7. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB11_final_projesi_atolyesi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-11/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(7)

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
p_cv = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X_train[ALL], y_train, cv=folds, method="predict_proba")[:, 1]
GRID = np.round(np.arange(0.01, 0.51, 0.01), 2)
threshold = max(t for t in GRID if (p_cv[y_train.to_numpy() == 1] >= t).mean() >= 0.95)
ruled_out = p_model < threshold
counts = {"TP": int((~ruled_out & (yt == 1)).sum()), "FN": int((ruled_out & (yt == 1)).sum()),
          "TN": int((ruled_out & (yt == 0)).sum()), "FP": int((~ruled_out & (yt == 0)).sum())}

print("eşik", threshold, "| çapraz doğrulama duyarlılığı", round(float((p_cv[y_train.to_numpy() == 1] >= threshold).mean()), 3))
print("çapraz doğrulama kalibrasyonu: sabit ve eğim", np.round(calibration(y_train.to_numpy(), p_cv), 3))
print(counts)
for name, k, n in (("duyarlılık", counts["TP"], counts["TP"] + counts["FN"]), ("özgüllük", counts["TN"], counts["TN"] + counts["FP"]),
                   ("negatif öngörü değeri", counts["TN"], counts["TN"] + counts["FN"])):
    print(f"{name}: {k}/{n} = {k / n:.3f}", np.round(wilson(k, n), 3))
print("dışlanan", int(ruled_out.sum()), "/", len(yt))

**Python notu.** `cross_val_predict(..., cv=folds, method="predict_proba")[:, 1]` işlem hattını her seferinde eğitim hastalarının beşte dördünde olmak üzere beş kez kurar ve her hasta için o hastayı dışarıda bırakan kurulumun olasılığını döndürür. `max(t for t in GRID if ...)` *üreteç ifadesi* ızgarayı dolaşır ve koşulu sağlayan en yüksek eşiği tutar. `p_model < threshold`, `True` ve `False` değerlerinden oluşan bir dizi verir; `~ruled_out` bu diziyi tersine çevirir. `&` bu tür iki diziyi hasta hasta birleştirir; böylece `(ruled_out & (yt == 1)).sum()` eşiğin altındaki hastalıklı hastaları sayar.

In [ ]:
ts = np.round(np.arange(0.01, 0.51, 0.01), 2)
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.2), sharey=True)
for ax, p, y, title in ((axes[0], p_cv, y_train.to_numpy(), "eğitim, çapraz doğrulama"), (axes[1], p_model, yt, "test")):
    ax.plot(ts, [(p[y == 1] >= t).mean() for t in ts], color="#9B1B1B", label="duyarlılık")
    ax.plot(ts, [(p < t).mean() for t in ts], color="#1F5F8B", label="dışlanan pay")
    ax.axhline(0.95, color="#8A8A8A", linestyle="--", linewidth=1)
    ax.axvline(threshold, color="#231A1A", linestyle=":", linewidth=1)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("eşik")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Alıştırma G.** Çapraz doğrulamalı eğitim hastalarında 0,90'lık bir duyarlılık hedefi hangi eşiği verirdi?

In [ ]:
veri = alistirma_tablosu("G")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma G", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = float(max(t for t in GRID if (p_cv[y_train.to_numpy() == 1] >= t).mean() >= 0.90))
teach.check("Alıştırma G", cevap)

## 8. Model neyi kullanıyor?

*Ders: [Model neyi kullanıyor ve nerede yanılabilir?](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-11/ders.html#what-the-model-uses-and-where-it-may-fail)*

Tam model standartlaştırılmış sütunlarla kurulan bir lojistik regresyondur; bu yüzden katsayıları birbiriyle karşılaştırılabilir ve bir sütunun bir hastanın log-odds değerine katkısı, katsayısı ile hastanın z-skorunun çarpımıdır. Bu, 9. haftada görülen kapalı formdur. Bölüm, en düşük olasılığa sahip test hastasını, yani haftanın vakası olan 260 numaralı hastayı açıklar.

**Formüllerden koda.** Standart sapma başına katsayılar: `coefficients`. Bir hastanın katkıları: `contributions`. En düşük olasılığa sahip hasta: `low`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 8. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB11_final_projesi_atolyesi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-11/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(8)

In [ ]:
scaler, logistic = model.named_steps["standardscaler"], model.named_steps["logisticregression"]
coefficients = pd.Series(logistic.coef_[0], index=ALL)


def contributions(patient):
    """Her sütunun hastanın log-odds değerine kattığı değeri hesaplar; bu değer katsayı ile z-skorunun çarpımıdır (9. haftanın kapalı formu)."""
    z = (patient[ALL].to_numpy(dtype=float) - scaler.mean_) / scaler.scale_
    return pd.Series(logistic.coef_[0] * z, index=ALL)

print(coefficients.sort_values(key=abs, ascending=False).round(3))
low = X_test.iloc[[int(np.argmin(p_model))]]
c_low = contributions(low.iloc[0])
eta = logistic.intercept_[0] + c_low.sum()
print("hasta", low.index[0], "| sabit terim", round(logistic.intercept_[0], 3), "| katkıların toplamı", round(c_low.sum(), 3),
      "| log-odds", round(eta, 3), "| olasılık", round(1 / (1 + np.exp(-eta)), 3), round(float(p_model.min()), 3))
fig, ax = plt.subplots(figsize=(7.2, 3.6))
order_ = c_low.sort_values().index
ax.barh(order_, c_low[order_], color=["#1F5F8B" if v < 0 else "#9B1B1B" for v in c_low[order_]])
ax.axvline(0, color="#231A1A", linewidth=1)
ax.set_xlabel("log-odds değerine katkı")
ax.set_title(f"Hasta {low.index[0]}", fontsize=9)
plt.tight_layout()
plt.show()

**Python notu.** `model.named_steps["standardscaler"]` ve `model.named_steps["logisticregression"]` işlem hattının iki adımına adlarıyla ulaşır. `scaler.mean_` ve `scaler.scale_`, ölçekleyicinin eğitim hastalarından öğrendiği ortalamalar ve standart sapmalardır; böylece `z` hastanın her sütundaki z-skoru olur. Çift köşeli parantezle yazılan `X_test.iloc[[k]]` tek satırlık bir tablo tutar, `.iloc[0]` de bu satırı bir seri olarak alır. `sort_values(key=abs, ascending=False)` katsayıları işaretlerinden bağımsız olarak büyüklüklerine göre sıralar.

**Alıştırma H.** 260 numaralı hasta, öteki bütün değerleri aynı kalmak üzere erkek olsaydı model ona hangi olasılığı verirdi? Üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("H")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma H", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
man = low.copy()
man["sex"] = 1
cevap = round(float(model.predict_proba(man[ALL])[:, 1][0]), 3)
teach.check("Alıştırma H", cevap)

## 9. Model nerede yanılabilir?

*Ders: [Model neyi kullanıyor ve nerede yanılabilir?](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-11/ders.html#what-the-model-uses-and-where-it-may-fail)*

Bir dışlama kullanıldığı her grupta güvenli olmalıdır. Bölüm, test kümesindeki kadınlar ve erkekler için duyarlılığı ve dışlanan hastaları, duyarlılığın Wilson aralığının alt sınırıyla birlikte yeniden hesaplar.

**Dersten koda.** Gruplar: `groups`. Grupların tablosu: `subgroups`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 9. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB11_final_projesi_atolyesi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-11/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(9)

In [ ]:
groups = {"kadınlar": X_test["sex"].to_numpy() == 0, "erkekler": X_test["sex"].to_numpy() == 1}
rows = []
for name, g in groups.items():
    k_tn, n_neg = int((ruled_out & (yt == 0) & g).sum()), int(((yt == 0) & g).sum())
    k_tp, n_pos = int((~ruled_out & (yt == 1) & g).sum()), int(((yt == 1) & g).sum())
    rows.append({"grup": name, "hasta": int(g.sum()), "hastalıklı": n_pos,
                 "duyarlılık": k_tp / n_pos, "alt sınır": wilson(k_tp, n_pos)[0],
                 "dışlanan": int((ruled_out & g).sum())})
subgroups = pd.DataFrame(rows)

print(subgroups.round(3).to_string(index=False))

**Python notu.** `groups`, bir grubun adından test hastaları üzerinde `True` ve `False` değerlerinden oluşan bir maskeye giden bir sözlüktür. Döngü her grup için hastalığı olup eşiğin üstünde kalan hastaları (`k_tp`) grubun hastalığı olan bütün hastalarına (`n_pos`) göre hesaplar; `wilson(k_tp, n_pos)[0]` de aralığın alt sınırını alır. Aynı döngü bir yaş grubu, bir hastane ya da bir dönem gibi her gruplama için çalışır.

**Gözlemle ve düşün.** Hastalığı olan hiçbir kadın kaçırılmaz; buna rağmen kadınlarda duyarlılığın alt sınırı yaklaşık 0,65 düzeyindedir. 0,95'lik bir alt sınır göstermek için bir doğrulamanın hastalığı olan kaç kadına, kadınların üçte birinde hastalık olsaydı toplam kaç kadına ihtiyacı olurdu?

**Alıştırma I.** Dışlama eşiği 60 ya da daha yaşlı test hastalarından kaçını anjiyografiden kurtarır?

In [ ]:
veri = alistirma_tablosu("I")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma I", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = int((ruled_out & (X_test["age"].to_numpy() >= 60)).sum())
teach.check("Alıştırma I", cevap)

## 10. Raporun sonuç tablosu

*Ders: [TRIPOD+AI ile raporlama](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-11/ders.html#reporting-with-tripod-ai)*

TRIPOD+AI her başarım ölçütünü, hem bütün hastalarda hem de modelin kullanılacağı gruplarda belirsizliğiyle birlikte ister. Bölüm çözümlü projenin bütün sonuçlarını aralıklarıyla birlikte tek bir tabloda toplar, tabloyu `final_project/results/results.csv` dosyasına yazar ve slaytlar için bir grafik olarak çizer.

**Dersten koda.** Raporun satırları: `results`. Tablo: `table`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 10. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB11_final_projesi_atolyesi.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-11/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(10)

In [ ]:
project = pathlib.Path("final_project")
for name in ["results", "figures"]:
    (project / name).mkdir(parents=True, exist_ok=True)
auc_ref, auc_model = roc_auc_score(yt, p_ref), roc_auc_score(yt, p_model)
results = [("referansın AUC değeri", "tümü", auc_ref, *interval(boot["a"])),
           ("modelin AUC değeri", "tümü", auc_model, *interval(boot["b"])),
           ("AUC farkı", "tümü", auc_model - auc_ref, *interval(boot["difference"])),
           ("kalibrasyon eğimi", "tümü", calibration(yt, p_model)[1], *slope_interval(yt, p_model))]
sex = X_test["sex"].to_numpy()
for group, g in (("tümü", np.ones(len(yt), dtype=bool)), ("kadınlar", sex == 0), ("erkekler", sex == 1)):
    for name, k, n in (("duyarlılık", int((~ruled_out & (yt == 1) & g).sum()), int(((yt == 1) & g).sum())),
                       ("özgüllük", int((ruled_out & (yt == 0) & g).sum()), int(((yt == 0) & g).sum())),
                       ("negatif öngörü değeri", int((ruled_out & (yt == 0) & g).sum()), int((ruled_out & g).sum()))):
        results.append((name, group, k / n, *wilson(k, n)))
table = pd.DataFrame(results, columns=["sonuç", "grup", "değer", "alt sınır", "üst sınır"]).round(3)
table.to_csv(project / "results" / "results.csv", index=False)

print(table.to_string(index=False))
plot = table[table["sonuç"] != "kalibrasyon eğimi"]
fig, ax = plt.subplots(figsize=(8.4, 4.6))
ypos = np.arange(len(plot))[::-1]
ax.errorbar(plot["değer"], ypos, xerr=[plot["değer"] - plot["alt sınır"], plot["üst sınır"] - plot["değer"]],
            fmt="o", color="#1F5F8B", ecolor="#8A8A8A", capsize=3)
ax.set_yticks(ypos)
ax.set_yticklabels([f"{r}, {g}" for r, g in zip(plot["sonuç"], plot["grup"])], fontsize=8)
ax.axvline(0, color="#231A1A", linewidth=0.8)
ax.set_xlim(-0.1, 1.05)
ax.set_title("Çözümlü projenin sonuçları ve yüzde 95'lik aralıkları (UCI Heart Disease, Cleveland, CC BY 4.0)", fontsize=9)
plt.tight_layout()
fig.savefig(project / "figures" / "results.png", dpi=150)
plt.show()

**Python notu.** `*interval(boot["a"])` aralığın iki ucunu demetin içine açar; böylece her satır beş değer taşır. İç içe iki döngü üç grubu ve üç oranı dolaşır ve her satır kendi Wilson aralığını alır. `pd.DataFrame(results, columns=[...])` beş sütunu adlandırır, `to_csv(..., index=False)` de tabloyu satır numaraları olmadan yazar. `plt.errorbar` her değeri aralığıyla birlikte çizer; `fig.savefig(...)` grafiği slaytlar için projenin klasöründe saklar.

**Alıştırma J.** 0,26 eşiğinde negatif öngörü değerinin Wilson aralığının alt sınırı nedir? Üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("J")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma J", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
out_26 = p_model < 0.26
cevap = round(wilson(int((out_26 & (yt == 0)).sum()), int(out_26.sum()))[0], 3)
teach.check("Alıştırma J", cevap)

## Final projesinin araç takımı ve seçeneklerin başlangıç noktaları

Bu haftanın denetimleri Cleveland verisine bağlı değildir. Aynı uzunlukta üç diziye ihtiyaç duyarlar; bunlar test hastalarının sonuçları, basit bir referansın olasılıkları ve projenin modelinin olasılıklarıdır. Aşağıdaki hücre bunları her fonksiyon için bir docstring içeren `final_project/code/project_tools.py` modülüne yazar; bir sonraki hücre de modülü çözümlü projede bir kez çalıştırır. Bu hücrenin sayıları 5-9. bölümlerin sayılarıdır.

| Seçenek | Başlangıç | Sonuç `y` | Referans `p_reference` | Model `p_model` |
|---|---|---|---|---|
| A. EKG | 1. ve 5. haftaların defterleri | Test EKG'lerinin bir ritim ya da iskemi etiketi | Tek bir ölçümle kurulan bir kural, örneğin QRS süresi ya da ST yükselmesi | 5. haftanın ağı |
| B. Risk | 2. ve 9. haftaların defterleri ve bu defter | Hastalık ya da izlemde bir olay | Kliniğin zaten kullandığı değişkenlerle kurulan bir model | Projenin modeli |
| C. Görüntüleme | 3., 4. ve 7. haftaların defterleri | Düşük ejeksiyon fraksiyonu ya da görüntülerin bir tanısı | Uzmanın ölçümü ya da daha basit bir ağ | Projenin ağı |
| D. Metin | 6. haftanın defteri | Sentetik rapordaki bulgu, sanal kliniğin gerçeğine göre | Anahtar kelime araması | Projenin bilgi çıkarımı; doğruluğu için `wilson` ile |
| E. Füzyon | 8. haftanın defteri | Sanal kliniğin hasta kaydındaki sonuç | Tek bir veri türüyle kurulan en iyi model | Bunları birleştiren model |
| F. Karar desteği | 9. ve 10. haftaların defterleri | Kullanıcı testindeki kayıtların gerçeği | Prototipin 1.0 sürümü | İyileştirilmiş sürüm |
| G. Kendi sorusu | Herhangi bir hafta | Analizden önce tanımlanmış klinik sonucu | Kliniğin şimdi yaptığı, bir skor ya da bir kural olarak | Kurumun bilgisayarında projenin modeli |

In [ ]:
pathlib.Path("final_project/code").mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile final_project/code/project_tools.py
"""Final projesinin araç takımı, 11. haftanın denetimlerini iki sonuçlu her model için içerir.

Her fonksiyon aynı uzunlukta üç diziyle çalışır: y sonuçları (0 ya da 1), p_reference basit bir
referansın olasılıklarını, p_model de projenin modelinin olasılıklarını tutar.
"""
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


def wilson(k, n, z=1.96):
    """n içinden k oranının Wilson aralığını verir."""
    p = k / n
    centre = (p + z ** 2 / (2 * n)) / (1 + z ** 2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / (1 + z ** 2 / n)
    return centre - half, centre + half


def bootstrap_difference(y, p_reference, p_model, n_boot=2000, seed=2026):
    """Hastaların yeniden örneklemlerinde referansın ve modelin AUC değerlerini ve farklarını hesaplar."""
    y, a, b = np.asarray(y), np.asarray(p_reference), np.asarray(p_model)
    rng, rows = np.random.default_rng(seed), []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() < y[i].max():
            ra, rb = roc_auc_score(y[i], a[i]), roc_auc_score(y[i], b[i])
            rows.append({"reference": ra, "model": rb, "difference": rb - ra})
    return pd.DataFrame(rows)


def percentile_interval(values, level=0.95):
    """Bootstrap değerlerinin yüzdelik aralığını verir."""
    return tuple(np.percentile(values, [50 * (1 - level), 100 - 50 * (1 - level)]))


def calibration(y, p, steps=50):
    """Kalibrasyon sabitini (log-odds değeri offset olarak alınarak) ve eğimini Newton adımlarıyla hesaplar."""
    y, lp = np.asarray(y, dtype=float), np.log(np.asarray(p) / (1 - np.asarray(p)))
    a = 0.0
    for _ in range(steps):
        q = 1 / (1 + np.exp(-(a + lp)))
        a += (y - q).sum() / (q * (1 - q)).sum()
    Z, b = np.c_[np.ones_like(lp), lp], np.zeros(2)
    for _ in range(steps):
        q = 1 / (1 + np.exp(-Z @ b))
        b += np.linalg.solve(Z.T @ (Z * (q * (1 - q))[:, None]), Z.T @ (y - q))
    return float(a), float(b[1])


def rule_out_threshold(p_cv, y, target=0.95, grid=np.round(np.arange(0.01, 0.51, 0.01), 2)):
    """Çapraz doğrulamalı eğitim olasılıklarında duyarlılığı en az target olan ve ızgaradaki en yüksek eşiği bulur."""
    p_cv, y = np.asarray(p_cv), np.asarray(y)
    return float(max(t for t in grid if (p_cv[y == 1] >= t).mean() >= target))


def rule_out_table(y, p, threshold, groups=None):
    """Bir eşikte duyarlılığı, özgüllüğü ve negatif öngörü değerini Wilson aralıklarıyla birlikte bütün hastalarda ve gruplarda verir."""
    y, out = np.asarray(y), np.asarray(p) < threshold
    groups = {"tümü": np.ones(len(y), dtype=bool), **(groups or {})}
    rows = []
    for name, g in groups.items():
        for result, k, n in (("duyarlılık", (~out & (y == 1) & g).sum(), ((y == 1) & g).sum()),
                             ("özgüllük", (out & (y == 0) & g).sum(), ((y == 0) & g).sum()),
                             ("negatif öngörü değeri", (out & (y == 0) & g).sum(), (out & g).sum())):
            if n > 0:
                rows.append({"sonuç": result, "grup": name, "k": int(k), "n": int(n), "değer": k / n,
                             "alt sınır": wilson(k, n)[0], "üst sınır": wilson(k, n)[1]})
    return pd.DataFrame(rows)

In [ ]:
sys.path.insert(0, str(pathlib.Path("final_project/code").resolve()))
import project_tools as tools
importlib.reload(tools)
b2 = tools.bootstrap_difference(yt, p_ref, p_model)
print("AUC farkı", np.round(tools.percentile_interval(b2["difference"]), 3))
print("kalibrasyon", np.round(tools.calibration(yt, p_model), 3))
t2 = tools.rule_out_threshold(p_cv, y_train, target=0.95)
print("eşik", t2)
print(tools.rule_out_table(yt, p_model, t2, {"kadınlar": X_test["sex"].to_numpy() == 0, "erkekler": X_test["sex"].to_numpy() == 1}).round(3).to_string(index=False))

**Python notu.** `%%writefile final_project/code/project_tools.py` ile başlayan bir hücre çalıştırılmaz: Jupyter hücrenin metnini dosyaya yazar ve dosya artık küçük bir modül tutar. `sys.path.insert(0, ...)` modülün klasörünü Python'un modül aradığı yerlere ekler; `import project_tools as tools` modülü kısa `tools` adıyla okur; `importlib.reload(tools)` da dosyayı değiştirdikten sonra modülü yeniden okur. Fonksiyonlar diziler ve maske sözlükleri aldığı için aynı beş satır her seçeneğin modelini değerlendirir.

## Alanlara göre uygulama görevleri

| Alan | Görev |
|---|---|
| Koroner arter hastalığı ve görüntüleme | Verileriniz izin veriyorsa çözümlü projenin sonucunu kendi hastalık tanımınızla, örneğin yüzde 70'lik bir daralmayla ya da görüntülemedeki iskemiyle değiştirin ve 4-9. bölümleri yeniden çalıştırın. Eşiğin ve dışlanan hastaların nasıl değiştiğini bildirin. |
| Kalp yetersizliği | İzlemde bir olayı tahmin eden bir model için sonucun zaman ufkunu analizden önce yazın ve modelin erken kontrole çağırmayacağı hastalar için `tools.rule_out_table` fonksiyonunu kullanın. |
| Elektrofizyoloji | 10. haftanın AF modelini F seçeneği olarak alın: Aynı test dakikalarında 1.0 sürümü referans, 1.1 sürümü model olsun; AUC değerlerinin farkını bootstrap aralığıyla birlikte verin. |
| Ekokardiyografi | Düşük ejeksiyon fraksiyonunu işaretleyen bir ağ için uzmanın ölçümünü referans olarak kullanın ve düşük görüntü kalitesine sahip hastalardaki duyarlılığı ayrı bir grup olarak bildirin. |
| Acil servis | Acil serviste bir dışlama hastaları eve gönderir. Alt sınırı 0,99 olan bir negatif öngörü değeri için bir doğrulamanın büyüklüğünü planlayın ve servisinizin bir yılda gördüğü hastalarla karşılaştırın. |
| Kalite ve hasta güvenliği | Modeliniz kullanıma girdikten sonra izleme kuralını yazın: Her üç ayda bir hangi hastalarda ölçülen hangi kalibrasyon eğimi kliniğin modeli durdurup gözden geçirmesini gerektirir? |
| Araştırma ve etik | Modelinizin sessiz çalıştırılması için etik kurula yapılacak başvurunun paragrafını yazın. Paragraf verileri, verilerin nerede kaldığını, çıktıyı kimin gördüğünü, hastaların neden etkilenmediğini ve büyüklüğüyle birlikte analiz planını anlatmalıdır. |

## Defterden sonra

Bu defter bir final projesini verilerinin denetiminden raporunun sonuç tablosuna ve her seçenek için çalışan bir araç takımına kadar götürdü. Her katılımcının final projesi aynı adımları kendi kliniğinin bir sorusuyla izler; oturum da sunumlar ve sertifikalarla kapanır. On bir haftanın bütün malzemeleri açık kalır: Her hafta kendi çalışma yoluyla yeniden izlenebilir.

<div align="center">

---

**Prof. Dr. Utku Köse**<br>
Profesör, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi, Isparta, Türkiye<br>
Kurucu Müdür, Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM), Süleyman Demirel Üniversitesi<br>
Bilgisayar Bilimleri Anabilim Dalı Başkanı, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi<br>
Diğer kurumlar: University of North Dakota (ABD), Universidad Panamericana (Meksiko, Meksika), Vel Tech University (Chennai, Hindistan)<br>
IEEE Kıdemli Üyesi, ACM Profesyonel Üyesi

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı, Türk Kardiyoloji Derneği. İlk sürüm: Kasım 2026. İçerik CC BY 4.0, kod MIT.*

</div>